<a href="https://colab.research.google.com/github/GuangyiLiu123/MyXPCS/blob/main/Guangyi_Copy_of_XPCS_FNS_Feb_2026_Francis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OVERVIEW

## Project Information

**Title:** XPCS Data Analysis for FNS

**Project:** XPCS Pipelines

**Version:** v4.3, July 2025

**Contributors:**
- Wei "Francis" He (wehe@ucsd.edu / francisho@lbl.gov)
- Dr. Rourav Basak (robasak@ucsd.edu)
- Robin Glefke (rglefke@ucsd.edu)
- Dr. Shan Wu
**Acknowledgement:**
- The original version is adapted from the analysis scipt provided at BL 7.0.1.1 by Dr. Sophie Morley (smorley@lbl.gov)
- The two-time correlation function calculation is adapted from codes provided by Dr. Margaret McCarter (mmccarter@lbl.gov)
- The speckle detection and selection feature is inspired and assisted by Dr. Ahmad Ikhwan Us Saleheen (aiussaleheen@lbl.gov)

---

## Experiment Info

**Beamline:** 7.0.1.1 (COSMIC) @ ALS

**Experiment Date:** April 2025

**Experimenter:** Dr. Shan Wu (swu13@scu.edu),
Dr. Rourav Basak (robasak@ucsd.edu)

**Experiment Log:** [COSMIC_2025_April](https://docs.google.com/document/d/1KANiAYFgllAaKO1rHyXqTlFeqeoBkAwAPV466V5W0hk/edit?tab=t.0)

**Local contact**: Dr. Sophie Morely (smorley@lbl.gov)

---

## Description

This notebook implements a comprehensive XPCS analysis pipeline for studying the temperature-dependent dynamics of the magnetic order in Fe$_x$NbS$_2$ ($x=0.323$), across different temperature regimes near magnetic phase transitions.


**Key Components:**
- **XPCS data manager:** Centralized system for organizing and managing multi-temperature datasets
- **Correlation analysis:** Calculate g₂(τ) functions using efficient FFT and multitau methods
- **Temperature series analysis:** Systematic comparison of dynamics across temperature ranges
- **Dynamics fitting:** Extract relaxation times and critical behavior parameters

**Main Expected Outputs:**
- $g_2(\tau)$ correlation functions for each temperature condition
- Temperature-dependent relaxation time analysis
- Critical behavior characterization near phase transitions

**Scientific Goals:**
- Characterize magnetic fluctuation dynamics across phase transitions
- Identify critical slowing down near magnetic ordering temperatures
- Quantify magnetic correlation lengths and characteristic timescales

# SETUP

This setup section configures the analysis environment optimized for Google Colab, including Google Drive integration for data access, conda package management for scientific computing libraries, and import of all required packages for XPCS analysis with version tracking for reproducibility.

Key Setup Components:
- Google Drive mounting for seamless data access
- condacolab installation for advanced package management
- Scientific libraries import (scikit-beam, h5py, lmfit, scipy, etc.)
- Analysis functions definition for XPCS correlation calculations
- Data manager initialization for organizing multi-temperature datasets


## Connect Google Drive and Install condacolab

Establish data access and prepare the Python environment for scientific computing packages that are not available in the default Colab installation.


In [ ]:
# Mount Google Drive for data access
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted")
print("📁 Available drives:")
!ls /content/drive/

In [ ]:
# Install condacolab - conda package manager designed specifically for Google Colab
# This enables installation of scientific packages with complex dependencies
!pip install -q condacolab
import condacolab
condacolab.install()

print("\n✅ condacolab installed!")
print("⚠️ RUNTIME WILL RESTART AUTOMATICALLY")
print("⚠️ After restart, continue with NEXT CELL")

In [ ]:
import sys
import subprocess

print("=== PYTHON ENVIRONMENT DIAGNOSTIC ===")
print(f"Active Python version: {sys.version}")
print(f"Python executable: {sys.executable}")

# Check which python conda will use
result = subprocess.run(['which', 'python'], capture_output=True, text=True)
print(f"Default python command points to: {result.stdout.strip()}")

# Check if conda environment is active
result = subprocess.run(['conda', 'info', '--envs'], capture_output=True, text=True)
print("\nConda environments:")
print(result.stdout)

## Install Packages and Import Libraries

Install core packages for XPCS analysis including scikit-beam for correlation calculations and supporting libraries. Import all required libraries with version information for reproducibility.

**Note:** Run this section only after the automatic restart.


In [ ]:
# Install core XPCS analysis packages
!conda install -c conda-forge numpy scipy matplotlib h5py -y
!conda install -c conda-forge scikit-beam lmfit ipywidgets -y

In [ ]:
# Install optional packages for specialized data formats
!conda install -c conda-forge tifffile nexusformat -y

In [ ]:
import sys

print("=== VERSION MISMATCH DIAGNOSTIC ===")
print(f"Current Python runtime: {sys.version}")
print(f"Python executable: {sys.executable}")
print()

# Check where packages are installed
!conda list | head -20

print("\nLet's see the specific Python versions in package names:")
!conda list | grep "py3"

print(f"\nPython version in sys: {sys.version_info.major}.{sys.version_info.minor}")

**Issue:** Python version mismatch between conda (3.11) and Colab runtime (3.12) causing import failures for scikit-beam and related packages.

**Temporary solution: Change runtime type to 2025.07.** See [this FAQ](https://research.google.com/colaboratory/runtime-version-faq.html) for more info.

In [ ]:
# Import core libraries
import numpy as np
import scipy
import scipy.fft
from scipy import stats
from scipy.signal import find_peaks   # To locate peaks in plots

# Data handling
import h5py                           # HDF5 file format
import os
import pickle                         # For data manager storage
import re
import time as ttime
import warnings
from datetime import datetime
from typing import Dict, List, Optional, Any
from glob import glob
from os.path import basename, getctime

# Visualization
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches  # To display ROI on detector image
import ipywidgets as widgets          # Interactive display
from IPython.display import display, clear_output

# XPCS analysis
import skbeam
import skbeam.core.correlation as corr
from skbeam.core.correlation import two_time_corr, two_time_state_to_results
import skbeam.core.roi as roi
import skbeam.core.utils as utils

# Fitting
import lmfit                          # Non-linear curve fitting
from lmfit import Model               # Import lmfit Model for fitting

# Optional libraries (with error handling)
try:
    import tifffile
    tifffile_version = tifffile.__version__
    tifffile_available = True
except ImportError:
    tifffile_version = "Not installed"
    tifffile_available = False

try:
    from nexusformat.nexus import *
    nexusformat_available = True
except ImportError:
    nexusformat_available = False

# Verify installation
print("Packages installed!")
print("-" * 30)
print(f"NumPy:          {np.__version__}")
print(f"SciPy:          {scipy.__version__}")
print(f"Matplotlib:     {matplotlib.__version__}")
print(f"h5py:           {h5py.__version__}")
print(f"scikit-beam:    {skbeam.__version__}")
print(f"lmfit:          {lmfit.__version__}")
print(f"ipywidgets:     {widgets.__version__}")

# Optional packages
if tifffile_available:
    print(f"tifffile:       {tifffile_version}")
else:
    print(f"tifffile:       Not installed (install with: !conda install -c conda-forge tifffile -y)")

if nexusformat_available:
    print(f"nexusformat:    Available")
else:
    print(f"nexusformat:    Not installed (install with: !conda install -c conda-forge nexusformat -y)")

## Data manager

To apply multi-temperature analysis, define a data manager that saves provessed ROI data and g2 results.


In [ ]:
class XPCSDataManager:
    """
    Simplified XPCS data manager using file_index as primary key.

    FUNCTION OUTLINE:
    ====================================================================

    Core Storage/Retrieval:
    ----------------------
    store_analysis()           : Store complete results for one measurement
    get_analysis()            : Retrieve all data by file_index
    get_roi_data()           : Retrieve specific ROI data
    add_correlation_analysis() : Add additional analysis to existing ROI

    Search & Filter:
    ---------------
    find_by_temperature()     : Find measurements near target temperature
    get_file_indices()       : Get all stored file indices
    get_temperatures()       : Get all unique temperatures
    get_summary()           : Display overview of stored data

    Persistence:
    -----------
    save_to_drive()          : Save to Google Drive
    load_from_drive()        : Load from Google Drive (class method)

    Core Philosophy:
    - file_index as unique primary key (no conflicts)
    - Temperature stored in metadata (searchable)
    - Multiple correlation analyses per ROI supported (dict keys)
    - Focused on storage/retrieval only
    - 9 core methods for complete functionality

    Storage Structure:
    - self.data = {file_index: {roi_name: {correlation_results_by_key}}}
    - self.metadata = {file_index: {temperature, timestamp, experiment_params}}
    - Correlation results stored as DICTIONARIES with analysis keys for easy access
    """

    def __init__(self, storage_path: Optional[str] = None):
        """Initialize simplified data manager."""
        if storage_path is None:
            storage_path = os.path.join(os.getcwd(), 'xpcs_analysis_results')

        self.storage_path = storage_path
        self.data = {}      # {file_index: {roi_name: analysis_results}}
        self.metadata = {}  # {file_index: {temperature, timestamp, file_info}}

        os.makedirs(storage_path, exist_ok=True)
        print(f"✅ XPCS Data Manager initialized (file_index-based)")
        print(f"📁 Storage: {storage_path}")

    # ========================================================================
    # CORE STORAGE/RETRIEVAL FUNCTIONS
    # ========================================================================

    def store_analysis(self,
                      file_index: int,
                      temperature: float,
                      roi_definitions: Dict[str, Dict],
                      two_time_g2_results: Dict[str, Dict],
                      one_time_corr_fft: Dict[str, Dict],
                      one_time_corr_multitau: Dict[str, Dict],
                      fit_results: Optional[Dict[str, Dict]] = None,
                      **metadata_kwargs) -> None:
        """
        Store complete analysis results for one measurement.

        STRUCTURE CLARIFICATION:
        - Each ROI gets its own analysis results
        - Each correlation type stored as DICTIONARY with analysis keys for easy access
        - Example: ROI 0 could have multiple multitau analyses: 'full_data', 'filtered_500_1500'

        Parameters:
        -----------
        file_index : int
            Unique file identifier (primary key)
        temperature : float
            Sample temperature in Kelvin
        roi_definitions : dict
            {'ROI 0': {'smd_shape': (h,w), 'coordinates': (x,y,w,h)}, ...}
        two_time_g2_results : dict
            {'ROI 0': {'g2_matrix': array, 'time_lags': array}, ...}
        one_time_corr_fft : dict
            {'ROI 0': {'full_data': {'g2': array, 'F': array, 'time_lags': array}}, ...}
        one_time_corr_multitau : dict
            {'ROI 0': {
                'full_data': {'g2': array, 'F': array, 'lags': array, 'start_frame': 0, 'end_frame': -1},
                'filtered_500_1500': {'g2': array, 'F': array, 'lags': array, 'start_frame': 500, 'end_frame': 1500}
            }, ...}
        fit_results : dict, optional
            {'ROI 0': {
                'full_data': {'A': float, 'Gamma': float, 'beta': float, ...},
                'filtered_500_1500': {'A': float, 'Gamma': float, 'beta': float, ...}
            }, ...}
        **metadata_kwargs : dict
            Essential metadata: data_shape, exposure_time, readout_time,
            beam_energy, sample_detector_distance, etc.
        """
        # Organize analysis data by ROI
        analysis_results = {}
        for roi_name in roi_definitions.keys():
            analysis_results[roi_name] = {
                'roi_definition': roi_definitions[roi_name],
                'two_time_g2': two_time_g2_results.get(roi_name, {}),
                'one_time_corr_fft': one_time_corr_fft.get(roi_name, {}),
                'one_time_corr_multitau': one_time_corr_multitau.get(roi_name, {}),
                'fit_results': fit_results.get(roi_name, {}) if fit_results else {}
            }

            # Validate that each ROI has corresponding correlation results
            if roi_name in two_time_g2_results:
                print(f"   {roi_name}: two-time g2 ✅")
            if roi_name in one_time_corr_fft and one_time_corr_fft[roi_name]:
                fft_keys = list(one_time_corr_fft[roi_name].keys())
                print(f"   {roi_name}: FFT correlation ✅ ({len(fft_keys)} analysis: {fft_keys})")
            if roi_name in one_time_corr_multitau and one_time_corr_multitau[roi_name]:
                multitau_keys = list(one_time_corr_multitau[roi_name].keys())
                print(f"   {roi_name}: Multitau correlation ✅ ({len(multitau_keys)} analysis: {multitau_keys})")
            if fit_results and roi_name in fit_results and fit_results[roi_name]:
                fit_keys = list(fit_results[roi_name].keys())
                print(f"   {roi_name}: Fit results ✅ ({len(fit_keys)} analysis: {fit_keys})")

        # Store analysis data
        self.data[file_index] = analysis_results

        # Store essential metadata with timestamp
        self.metadata[file_index] = {
            'temperature': temperature,
            'timestamp': datetime.now().isoformat(),
            'roi_count': len(analysis_results),
            'roi_names': list(analysis_results.keys()),
            **metadata_kwargs
        }

        print(f"✅ Stored analysis for file_index {file_index}")
        print(f"   Temperature: {temperature:.1f}K, ROIs: {len(analysis_results)}")

        # Display essential metadata if provided
        if metadata_kwargs:
            key_params = ['data_shape', 'exposure_time', 'readout_time', 'beam_energy']
            displayed_params = {k: v for k, v in metadata_kwargs.items() if k in key_params}
            if displayed_params:
                print(f"   Key params: {displayed_params}")

    def get_analysis(self, file_index: int) -> Dict[str, Any]:
        """
        Retrieve complete analysis results by file_index.

        Returns:
        --------
        dict : {roi_name: analysis_results, 'metadata': metadata_dict}
        """
        if file_index not in self.data:
            raise ValueError(f"No data found for file_index {file_index}")

        return {
            **self.data[file_index],
            'metadata': self.metadata[file_index]
        }

    def get_roi_data(self, file_index: int, roi_name: str) -> Dict[str, Any]:
        """Retrieve specific ROI analysis results."""
        if file_index not in self.data:
            raise ValueError(f"No data found for file_index {file_index}")

        if roi_name not in self.data[file_index]:
            available_rois = list(self.data[file_index].keys())
            raise ValueError(f"ROI '{roi_name}' not found. Available: {available_rois}")

        return self.data[file_index][roi_name]

    def get_summary(self) -> None:
        """Display comprehensive summary of stored data."""
        if not self.data:
            print("📊 No analysis data stored")
            return

        print("📊 XPCS DATA MANAGER SUMMARY")
        print("=" * 50)
        print(f"Total measurements: {len(self.data)}")

        # Group by temperature for display
        temp_groups = {}
        for file_idx, meta in self.metadata.items():
            temp = meta['temperature']
            if temp not in temp_groups:
                temp_groups[temp] = []
            temp_groups[temp].append(file_idx)

        for temp in sorted(temp_groups.keys()):
            file_indices = temp_groups[temp]
            print(f"\n🌡️  {temp:.1f}K: {len(file_indices)} measurement(s)")

            for file_idx in sorted(file_indices):
                meta = self.metadata[file_idx]
                roi_count = meta['roi_count']
                timestamp = meta['timestamp'][:10]  # Just date
                print(f"   File {file_idx}: {roi_count} ROIs ({timestamp})")

    def add_correlation_analysis(self,
                                file_index: int,
                                roi_name: str,
                                correlation_type: str,
                                analysis_key: str,
                                correlation_result: Dict[str, Any]) -> None:
        """
        Add an additional correlation analysis to an existing ROI.

        Useful for adding filtered or windowed analyses without re-storing everything.

        Parameters:
        -----------
        file_index : int
            Existing measurement identifier
        roi_name : str
            Existing ROI name
        correlation_type : str
            'one_time_corr_fft' or 'one_time_corr_multitau'
        analysis_key : str
            Key for this analysis (e.g., 'filtered_500_1500', 'windowed_700_2000')
        correlation_result : dict
            New correlation result
        """
        if file_index not in self.data:
            raise ValueError(f"No data found for file_index {file_index}")

        if roi_name not in self.data[file_index]:
            available_rois = list(self.data[file_index].keys())
            raise ValueError(f"ROI '{roi_name}' not found. Available: {available_rois}")

        if correlation_type not in ['one_time_corr_fft', 'one_time_corr_multitau']:
            raise ValueError(f"Invalid correlation_type. Use 'one_time_corr_fft' or 'one_time_corr_multitau'")

        # Add with new analysis key
        self.data[file_index][roi_name][correlation_type][analysis_key] = correlation_result

        # Update metadata timestamp
        self.metadata[file_index]['last_updated'] = datetime.now().isoformat()

        all_keys = list(self.data[file_index][roi_name][correlation_type].keys())
        print(f"✅ Added '{analysis_key}' {correlation_type} to {roi_name}")
        print(f"   All {correlation_type} keys for {roi_name}: {all_keys}")

    # ========================================================================
    # SEARCH & FILTER FUNCTIONS
    # ========================================================================

    def get_file_indices(self) -> List[int]:
        """Get all stored file indices."""
        return sorted(self.data.keys())

    def get_temperatures(self) -> List[float]:
        """Get all unique temperatures."""
        temps = [meta['temperature'] for meta in self.metadata.values()]
        return sorted(set(temps))

    def find_by_temperature(self, target_temp: float, tolerance: float = 0.1) -> List[int]:
        """
        Find file indices with temperatures within tolerance of target.

        Returns:
        --------
        list : File indices sorted by proximity to target temperature
        """
        matches = []
        for file_idx, meta in self.metadata.items():
            temp_diff = abs(meta['temperature'] - target_temp)
            if temp_diff <= tolerance:
                matches.append((file_idx, temp_diff))

        # Sort by temperature proximity
        matches.sort(key=lambda x: x[1])
        return [file_idx for file_idx, _ in matches]

    # ========================================================================
    # PERSISTENCE
    # ========================================================================

    def save_to_drive(self, filename: str = 'xpcs_analysis.pkl',
                     drive_path: str = '/content/drive/MyDrive/') -> None:
        """Save to Google Drive with simplified structure."""
        filepath = os.path.join(drive_path, filename)

        save_data = {
            'data': self.data,
            'metadata': self.metadata,
            'storage_path': self.storage_path,
            'save_timestamp': datetime.now().isoformat(),
            'version': 'v2.0_file_index_based'
        }

        try:
            with open(filepath, 'wb') as f:
                pickle.dump(save_data, f)

            file_size_mb = os.path.getsize(filepath) / 1024 / 1024
            print(f"☁️  Saved to: {filepath}")
            print(f"💾 Size: {file_size_mb:.2f} MB")
            print(f"📊 Data: {len(self.data)} measurements")

        except FileNotFoundError:
            print("❌ Google Drive not mounted!")
            print("Run: from google.colab import drive; drive.mount('/content/drive')")

    @classmethod
    def load_from_drive(cls, filepath: str) -> 'XPCSDataManager':
        """Load from Google Drive."""
        try:
            with open(filepath, 'rb') as f:
                save_data = pickle.load(f)

            # Create new instance
            manager = cls(storage_path=save_data['storage_path'])

            # Restore data
            manager.data = save_data['data']
            manager.metadata = save_data['metadata']

            version = save_data.get('version', 'unknown')
            print(f"☁️  Loaded: {filepath}")
            print(f"📊 Data: {len(manager.data)} measurements")
            print(f"🏷️  Version: {version}")

            return manager

        except FileNotFoundError:
            print(f"❌ File not found: {filepath}")
            raise
        except Exception as e:
            print(f"❌ Error loading: {e}")
            raise


In [ ]:
# Convenience Functions

def setup_xpcs_manager(storage_path: Optional[str] = None) -> XPCSDataManager:
    """Quick setup for new XPCS data manager."""
    return XPCSDataManager(storage_path)

def load_xpcs_manager(filepath: str) -> XPCSDataManager:
    """Quick function to load data manager from drive."""
    return XPCSDataManager.load_from_drive(filepath)

### XPCS Data Manager Usage Guide


**Step 1: Initialize Data Manager**

```python
# INITIALIZE XPCS DATA MANAGER v2.0
print("Setting up XPCS Data Manager v2.0 ...\n")
results_path = os.path.join(Data_path, 'Francis_XPCS', 'Results_v2')

# Create results directory if it doesn't exist
os.makedirs(results_path, exist_ok=True)

# Initialize the new data manager
dm = setup_xpcs_manager(storage_path=results_path)
dm.get_summary()
```

**Step 2: Store Analysis Results for Each Measurement**

After you complete your analysis for one measurement, store the results:

```python
# Example: After analyzing file_index 19 (temperature 38.0K)
file_index = 19  # From your file selection
current_temperature = np.mean(temperature)  # 38.0K

# Prepare your data structures (these come from your analysis)
roi_definitions = {
    'ROI 0': {'smd_shape': smd_0.shape, 'coordinates': (roi_0_x, roi_0_y, roi_0_width, roi_0_height)},
    'ROI 1': {'smd_shape': smd_1.shape, 'coordinates': (roi_1_x, roi_1_y, roi_1_width, roi_1_height)},
    'ROI 2': {'smd_shape': smd_2.shape, 'coordinates': (roi_2_x, roi_2_y, roi_2_width, roi_2_height)}
}

two_time_g2_results = {
    'ROI 0': {'g2_matrix': g2_two_time_results[0][0], 'time_lags': time_lags_results[0]},
    'ROI 1': {'g2_matrix': g2_two_time_results[1][0], 'time_lags': time_lags_results[1]},
    'ROI 2': {'g2_matrix': g2_two_time_results[2][0], 'time_lags': time_lags_results[2]}
}

one_time_corr_fft = {
    'ROI 0': {'full_data': correlation_results[0]},
    'ROI 1': {'full_data': correlation_results[1]},
    'ROI 2': {'full_data': correlation_results[2]}
}

one_time_corr_multitau = {
    'ROI 0': {'full_data': multitau_results[0]},
    'ROI 1': {'full_data': multitau_results[1]},
    'ROI 2': {'full_data': multitau_results[2]}
}

# Store everything in the data manager
dm.store_analysis(
    file_index=file_index,
    temperature=current_temperature,
    roi_definitions=roi_definitions,
    two_time_g2_results=two_time_g2_results,
    one_time_corr_fft=one_time_corr_fft,
    one_time_corr_multitau=one_time_corr_multitau,
    fit_results=fit_results_processed,
    # Essential metadata
    data_shape=data.shape,
    exposure_time=exposure_time,
    readout_time=readout_time,
    beam_energy=np.mean(beamline_energy)
)

# Check what's stored
dm.get_summary()
```

**Step 3: Repeat for Other Measurements**

```python
# Analyze file_index 20 (40.0K), then store
dm.store_analysis(file_index=20, temperature=40.0, roi_definitions=..., ...)

# Analyze file_index 21 (38.5K - another 38K measurement!), then store  
dm.store_analysis(file_index=21, temperature=38.5, roi_definitions=..., ...)

# No more conflicts! Multiple measurements per temperature supported.
```

**Step 4: Save to Google Drive (Permanent Storage)**

```python
## Mount Google Drive first (run this once per Colab session)
from google.colab import drive
drive.mount('/content/drive')

# Save your data manager to Google Drive
dm.save_to_drive('xpcs_analysis_v2.pkl')

# Your data is now permanently saved!
```

**Step 5: Load from Google Drive (After Restarting Colab)**

```python
# After restarting Colab runtime...
# Mount Google Drive again
from google.colab import drive
drive.mount('/content/drive')

# Load your saved data manager
dm = load_xpcs_manager_v2('/content/drive/MyDrive/xpcs_analysis_v2.pkl')

# All your processed results are back!
dm.get_summary()
```

**Step 6: Retrieve and Use Your Processed Results**

Basic Retrieval by file_index
```python
# Get complete analysis for specific measurement
results_file_19 = dm.get_analysis(file_index=19)
metadata = results_file_19['metadata']
roi_0_data = results_file_19['ROI 0']

# Get specific ROI data
roi_0_from_file_19 = dm.get_roi_data(file_index=19, roi_name='ROI 0')
g2_data_multitau = roi_0_from_file_19['one_time_corr_multitau']['full_data']['g2']
F_data = roi_0_from_file_19['one_time_corr_multitau']['full_data']['F']
```

Temperature-Based Queries
```python
# Find all measurements near 38K (±0.1K tolerance)
files_near_38K = dm.find_by_temperature(38.0, tolerance=0.1)
print(f"Found {len(files_near_38K)} measurements near 38K: {files_near_38K}")

# Get available temperatures
all_temps = dm.get_temperatures()
print(f"Available temperatures: {all_temps}")
```

Cross-Temperature Comparison
```python
# Compare decay curves across temperatures for ROI 0
target_temperatures = [37.5, 38.0, 38.5, 39.0, 40.0]

plt.figure(figsize=(10, 6))
for temp in target_temperatures:
    file_indices = dm.find_by_temperature(temp, tolerance=0.2)
    if file_indices:
        roi_data = dm.get_roi_data(file_indices[0], 'ROI 0')
        multitau_data = roi_data['one_time_corr_multitau']['full_data']
        
        F_data = multitau_data['F'].flatten()
        time_lags = multitau_data['lags']
        
        plt.semilogx(time_lags, F_data, 'o-', label=f'{temp:.1f}K', markersize=2)

plt.xlabel('τ (s)')
plt.ylabel('F(τ)')
plt.title('Temperature Comparison - ROI 0')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
```

Cross-ROI Comparison (Same Measurement)
```python
# Compare different ROIs from same measurement
file_idx = 19
rois_to_compare = ['ROI 0', 'ROI 1', 'ROI 2']

plt.figure(figsize=(10, 6))
for roi_name in rois_to_compare:
    roi_data = dm.get_roi_data(file_idx, roi_name)
    multitau_data = roi_data['one_time_corr_multitau']['full_data']
    
    F_data = multitau_data['F'].flatten()
    time_lags = multitau_data['lags']
    
    plt.semilogx(time_lags, F_data, 'o-', label=roi_name, markersize=2)

plt.xlabel('τ (s)')
plt.ylabel('F(τ)')
plt.title('ROI Comparison - Single Measurement')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
```

**Step 7: Add Multiple Analyses per ROI**

Store Multiple Correlations at Once
```python
# Example: Store both full data and filtered data for ROI 0
multitau_multiple = {
    'ROI 0': {
        'full_data': {'g2': g2_full, 'F': F_full, 'lags': lags},
        'filtered_500_1500': {'g2': g2_filtered, 'F': F_filtered, 'lags': lags}
    }
}

dm.store_analysis(file_index=22, temperature=39.0, ...,
                 one_time_corr_multitau=multitau_multiple, ...)
```

Add Analysis to Existing Measurement
```python
# Add filtered analysis to existing measurement
filtered_result = {
    'g2': g2_filtered_roi0,
    'F': F_filtered_roi0,
    'lags': lags_filtered,
    'start_frame': 700,
    'end_frame': 2000
}

dm.add_correlation_analysis(
    file_index=19,
    roi_name='ROI 0',
    correlation_type='one_time_corr_multitau',
    analysis_key='filtered_700_2000',
    correlation_result=filtered_result
)

# Access the new analysis
roi_data = dm.get_roi_data(19, 'ROI 0')
filtered_data = roi_data['one_time_corr_multitau']['filtered_700_2000']
```

**Step 8: Advanced Usage**

Batch Processing
```python
# Process multiple files in a loop
file_indices_to_analyze = [15, 16, 17, 18, 19, 20, 21]

for file_idx in file_indices_to_analyze:
    print(f"\n--- Processing file_index {file_idx} ---")
    
    # Your analysis code here...
    # (load data, run correlation analysis, etc.)
    
    # Store results with file_index as primary key
    dm.store_analysis(
        file_index=file_idx,
        temperature=current_temp,
        roi_definitions=roi_definitions,
        two_time_g2_results=two_time_g2_results,
        one_time_corr_fft=one_time_corr_fft,
        one_time_corr_multitau=one_time_corr_multitau,
        data_shape=data.shape,
        exposure_time=exposure_time,
        readout_time=readout_time
    )
    
    # Save progress periodically
    if file_idx % 3 == 0:
        dm.save_to_drive(f'progress_file_{file_idx}.pkl')

# Final save
dm.save_to_drive('complete_analysis_v2.pkl')
```

Data Overview
```python
# Get overview of all stored measurements
dm.get_summary()

# Get detailed information
all_file_indices = dm.get_file_indices()
print(f"Stored measurements: {all_file_indices}")

for file_idx in all_file_indices:
    metadata = dm.metadata[file_idx]
    print(f"File {file_idx}: {metadata['temperature']:.1f}K, "
          f"{metadata['roi_count']} ROIs, {metadata['timestamp'][:10]}")
```


## Define Custom Functions

Implement specialized functions for XPCS correlation analysis and data processing. These functions are designed to be used repeatedly across different measurements without modification.

`interactive_frame_viewer`: Create streamlined interactive frame viewer with slider, navigation buttons, and direct frame input for exploring detector images.

`create_roi_slice`: Create and validate region of interest (ROI) slice for data analysis with boundary checking.


In [ ]:
def interactive_frame_viewer(data, vmin=0, vmax=1500, figsize=(8, 7), set_title='Detector Image'):
    """
    Create an interactive frame viewer with streamlined slider control.

    Parameters:
    -----------
    data : array-like
        3D array with shape (n_frames, height, width)
        Works for both full detector images and ROI data
    vmin, vmax : float
        Intensity scale limits for display
    figsize : tuple
        Figure size for the plot (width, height)
    set_title : str
        Base title for the images
    """

    # Create output widget to contain the plot
    output_widget = widgets.Output()

    # Function to update and display the plot
    def update_frame(frame_idx):
        with output_widget:
            clear_output(wait=True)

            # Create new figure for each update
            fig, ax = plt.subplots(figsize=figsize)
            im = ax.imshow(data[frame_idx], cmap='viridis', vmin=vmin, vmax=vmax)
            ax.set_xlabel('Pixel X')
            ax.set_ylabel('Pixel Y')
            ax.set_title(f'{set_title} - Frame {frame_idx} / {len(data)-1}')
            plt.colorbar(im, ax=ax, label='Intensity (counts)')
            plt.tight_layout()
            plt.show()

    # Create slider widget
    frame_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=len(data)-1,
        step=1,
        description='Frame:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='60%')
    )

    # Create navigation buttons
    prev_button = widgets.Button(description="◀ Prev", layout=widgets.Layout(width='80px'),
                                 tooltip="Previous frame")
    next_button = widgets.Button(description="Next ▶", layout=widgets.Layout(width='80px'),
                                 tooltip="Next frame")

    # Create text input for direct frame selection
    frame_input = widgets.IntText(
        value=0,
        min=0,
        max=len(data)-1,
        step=1,
        description="Go to:",
        layout=widgets.Layout(width='120px')
    )

    # Button event handlers
    def on_prev_click(b):
        new_frame = max(0, frame_slider.value - 1)
        frame_slider.value = new_frame
        frame_input.value = new_frame

    def on_next_click(b):
        new_frame = min(len(data) - 1, frame_slider.value + 1)
        frame_slider.value = new_frame
        frame_input.value = new_frame

    def on_text_change(change):
        if 'new' in change:
            frame_slider.value = change['new']

    # Connect event handlers
    prev_button.on_click(on_prev_click)
    next_button.on_click(on_next_click)
    frame_input.observe(on_text_change, names='value')

    # Connect slider to update function and sync with text input
    def on_slider_change(change):
        if 'new' in change:
            update_frame(change['new'])
            frame_input.value = change['new']

    frame_slider.observe(on_slider_change, names='value')

    # Create control panel
    controls = widgets.HBox([
        prev_button,
        next_button,
        frame_input,
        frame_slider
    ])

    # Display initial frame
    update_frame(0)

    # Display controls and output
    display(controls)
    display(output_widget)

    return None


In [ ]:
def create_roi_slice(x, y, width, height, data_shape, roi_name=None):
    """
    Create and validate a region of interest (ROI) slice for data analysis.

    This function creates numpy slice objects for extracting ROI data from
    detector images, with comprehensive boundary validation to prevent errors.

    Parameters:
    -----------
    x, y : int
        Top-left corner coordinates of the ROI (x=column, y=row)
    width, height : int
        ROI dimensions in pixels
    data_shape : tuple
        Shape of the data array (n_frames, height, width)
    roi_name : str, optional
        Name of the ROI for informative output (e.g., 'ROI 0', 'Background ROI')

    Returns:
    --------
    tuple
        Slice object for indexing the ROI: (slice(y, y+height), slice(x, x+width))
        Usage: roi_data = full_data[(..., *roi_slice)]

    Raises:
    -------
    ValueError
        If ROI extends beyond data boundaries or coordinates are invalid
    """

    _, data_height, data_width = data_shape

    # Validate ROI coordinates are non-negative
    if x < 0 or y < 0:
        raise ValueError(f"ROI coordinates must be non-negative, got x={x}, y={y}")

    # Validate ROI fits within data boundaries
    if x + width > data_width:
        raise ValueError(f"ROI width extends beyond detector: x + width = {x + width} > detector width {data_width}")

    if y + height > data_height:
        raise ValueError(f"ROI height extends beyond detector: y + height = {y + height} > detector height {data_height}")

    # Create ROI slice tuple
    roi_slice = (slice(y, y + height), slice(x, x + width))

    if roi_name:
        print(f"{roi_name} created: ({x}, {y}) with size {width}×{height}")
    else:
        print(f"ROI created: ({x}, {y}) with size {width}×{height}")

    return roi_slice

### Funtions to calculate correlation

We use two methods to calculate the correlation functions:

1. **FFT method**: Efficient O(n log n) algorithm for large datasets
2. **Multitau method** - Scikit-beam implementation with logarithmic time spacing

`calculate_corr_fft`: Calculate g₂(τ) and F(τ) using FFT method with uniform lag_time_s spacing - ideal for large datasets, and good for displaying.

`g2_scipy_fft`: Core FFT-based correlation calculation (provided by Sophie).

`calculate_corr_multitau`: Calculate g₂(τ) and F(τ) using multitau algorithm with logarithmic lag_time_s spacing - ideal for wide time ranges fitting.

`calculate_lag_steps`: Educational function showing multitau lag_time_s calculation logic - not used in main analysis workflow.

In [ ]:
def calculate_corr_fft(data, identifier=None, start_frame=0, end_frame=None, frame_time=None):
    """
    Calculate g2 correlation function and F(τ) using FFT method.

    The FFT method provides O(n log n) performance, making it ideal for large datasets
    with many frames. Use this method when you need all correlation lag times with
    uniform spacing and have sufficient computational resources.

    Parameters:
    -----------
    data : 3D array
        Data array (frames, height, width)
    identifier : str, optional
        Data identifier for labeling (e.g., 'ROI 0', '38K', 'Sample A', etc.)
    start_frame : int, optional
        Starting frame index (default: 0)
    end_frame : int, optional
        Ending frame index, None for all frames (default: None)
    frame_time : float, optional
        Time per frame in seconds. If None, uses (exposure_time + readout_time)/1000

    Returns:
    --------
    dict
        Dictionary containing:
        - 'g2': g2 correlation function
        - 'F': F(τ) intermediate scattering function
        - 'lag_time_s': time lag array in seconds
        - 'identifier': data identifier
        - 'method': 'fft'
        - 'start_frame': start frame used
        - 'end_frame': end frame used
        - 'description': human-readable description

    Example:
    --------
    >>> results = calculate_corr_fft(smd_0, identifier='ROI 0', start_frame=500, end_frame=2000)
    >>> g2 = results['g2']
    >>> F_tau = results['F']
    >>> lag_time_s = results['lag_time_s']
    """

    # Input validation
    if end_frame is None:
        end_frame = len(data)
    elif end_frame > len(data):
        raise ValueError(f"end_frame ({end_frame}) cannot be larger than total frames ({len(data)})")

    # Apply frame selection
    selected_data = data[start_frame:end_frame]

    # Calculate frame time if not provided
    if frame_time is None:
        frame_time = (exposure_time + readout_time) / 1000  # Convert ms to seconds

    print(f"FFT correlation calculation:")
    print(f"  Data shape: {selected_data.shape}")
    print(f"  Identifier: {identifier or 'None'}")
    print(f"  Frame time: {frame_time:.4f} s")

    # Calculate g2 using FFT method
    g2 = g2_scipy_fft(selected_data.astype('float64'))

    # Create time lag array
    lag_time_s = np.arange(len(g2)) * frame_time

    # Calculate F(τ) from g2
    F = np.sqrt((g2[1:] - 1) / (g2[1:10].mean() - 1))

    print(f"  Time range: {lag_time_s[0]:.3f} s to {lag_time_s[-1]:.1f} s")
    print(f"  Generated {len(g2)} correlation points")
    print(f"  Calculation complete ✅")

    description = f"{identifier}, frames {start_frame} to {end_frame-1} (out of {len(data)} frames)" if identifier is not None else f"frames {start_frame} to {end_frame-1} (out of {len(data)} frames)"

    results = {
        'g2': g2,
        'F': F,
        'lag_time_s': lag_time_s,
        'identifier': identifier,
        'method': 'fft',
        'start_frame': start_frame,
        'end_frame': end_frame,
        'description': description
    }

    return results


In [ ]:
def g2_scipy_fft(img_stack):
    """
    Calculate the two-time intensity correlation function g₂(τ) using Fast Fourier Transform.

    This function implements an FFT-based algorithm that scales as O(n log n) instead of
    the traditional O(n²) direct method, making it practical for large XPCS datasets
    with thousands of frames.

    Parameters:
    -----------
    img_stack : numpy.ndarray
        3D array of shape (n_frames, height, width) containing the time series of
        detector images. Data type should be float for optimal performance.

    Returns:
    --------
    numpy.ndarray
        1D array of g₂ correlation values as a function of time lag. The array length
        equals the number of input frames, with g₂[0] corresponding to zero lag.

    Algorithm Details:
    ------------------
    1. Zero-pad the input array to avoid circular correlation artifacts
    2. Compute FFT of the padded array
    3. Calculate correlation via inverse FFT of |FFT|²
    4. Normalize by the number of contributing pairs at each lag
    5. Compute ensemble average over all pixels
    6. Normalize by the square of the temporal average

    Performance Notes:
    ------------------
    - For n frames: Direct method ~ O(n²), FFT method ~ O(n log n)
    - Memory usage: ~4x input array size during computation
    - Recommended for datasets with >100 frames
    """

    print('Calculating the numerators via FFT')
    img_stack_padded = np.concatenate([img_stack, np.zeros_like(img_stack)],
                                      axis=0)
    img_stack_fft = scipy.fft.fft(img_stack_padded, axis=0)
    numerator_base = scipy.fft.ifft(img_stack_fft * img_stack_fft.conj(),
                                 axis=0)[:img_stack.shape[0]].real
    n_elements = (np.arange(img_stack.shape[0])+1)[::-1]
    numerator_base /= n_elements[:,None,None]

    # This is just an efficient way to calculate the denominator
    print('Calculating the denominators')
    lcumsum = np.roll(np.cumsum(img_stack, axis=0),1, axis=0)
    lcumsum[0,:,:] = 0
    rcumsum = np.roll(np.cumsum(img_stack[::-1], axis=0),1, axis=0)
    rcumsum[0,:,:] = 0
    denominator_base = (2 * np.sum(img_stack, axis=0)) - lcumsum - rcumsum
    n_elements = 2 * img_stack.shape[0] - 2*np.arange(img_stack.shape[0])
    denominator_base /= n_elements[:,None,None]
    # And we have our result!
    return numerator_base.mean(axis=(1,2)) / denominator_base.mean(axis=(1,2))**2


In [ ]:
def calculate_corr_multitau(data, identifier=None, start_frame=0, end_frame=None,
                           num_levels=6, num_bufs=12, frame_time=None):
    """
    Calculate g2 correlation function and F(τ) using multi-tau algorithm.

    The multi-tau method provides logarithmic time spacing and better statistics at
    longer lag times, making it ideal for studying dynamics over wide time ranges.
    Use this method when you need to cover many decades of time with good statistics.

    Parameters:
    -----------
    data : 3D array
        Data array (frames, height, width)
    identifier : str, optional
        Data identifier for labeling (e.g., 'ROI 0', '38K', 'Sample A', etc.)
    start_frame : int, optional
        Starting frame index (default: 0)
    end_frame : int, optional
        Ending frame index, None for all frames (default: None)
    num_levels : int, optional
        Number of correlation time levels (default: 6)
        Higher values = wider time range coverage
    num_bufs : int, optional
        Number of buffers per level (default: 12)
        Higher values = better statistics per level
    frame_time : float, optional
        Time per frame in seconds. If None, uses (exposure_time + readout_time)/1000

    Returns:
    --------
    dict
        Dictionary containing:
        - 'g2': g2 correlation function (first point removed)
        - 'F': F(τ) intermediate scattering function
        - 'lag_time_s': time lag array in seconds
        - 'identifier': data identifier
        - 'method': 'multitau'
        - 'start_frame': start frame used
        - 'end_frame': end frame used
        - 'num_points': number of correlation points
        - 'time_range': (min_lag, max_lag) in seconds
        - 'description': human-readable description

    Notes:
    ------
    For 2700 frames (~45 minutes at 1 sec/frame):
    - num_levels=6: Good balance of range vs statistics
    - num_levels=7-8: Wider time range if needed
    - num_bufs=12-20: Excellent statistics with your data amount

    Examples:
    ---------
    >>> # For ROI analysis
    >>> results = calculate_corr_multitau(smd_0, identifier='ROI 0')
    >>> # For temperature series
    >>> results = calculate_corr_multitau(data_38K, identifier='38K')
    """

    # Input validation
    if end_frame is None:
        end_frame = len(data)
    elif end_frame > len(data):
        raise ValueError(f"end_frame ({end_frame}) cannot be larger than total frames ({len(data)})")

    # Apply frame selection
    selected_data = data[start_frame:end_frame]

    # Calculate frame time if not provided
    if frame_time is None:
        frame_time = (exposure_time + readout_time) / 1000  # Convert ms to seconds

    print(f"Calculating one-time correlation via corr.multitau ...")
    print(f"  Data shape: {selected_data.shape}")
    print(f"  Identifier: {identifier or 'None'}")

    # Create mask (all pixels in region are included)
    mask = np.ones(selected_data.shape[1:], dtype=np.int64)

    # Calculate correlation using multi-tau algorithm
    g2n, lag_steps = corr.multi_tau_auto_corr(
        num_levels,
        num_bufs,
        mask,
        selected_data
    )

    # Convert lag steps to actual time using frame time
    lag_time_s = lag_steps * frame_time

    # Remove first point (zero lag) as is standard in XPCS
    g2_clean = g2n[1:]
    lag_time_s_clean = lag_time_s[1:]

    # Calculate normalized correlation (F function)
    # Using first 10 points for normalization
    F = np.sqrt((g2_clean - 1) / (g2_clean[:10].mean() - 1))

    print(f"  Lag steps: {lag_steps[1:6].tolist()} ... {lag_steps[-3:].tolist()} (frames)")
    print(f"  Time range: {lag_time_s_clean[0]:.3f} s to {lag_time_s_clean[-1]:.1f} s")
    print(f"  Calculation complete.\n")

    description = f"{identifier}, frames {start_frame} to {end_frame-1} (out of {len(data)} frames)" if identifier is not None else f"frames {start_frame} to {end_frame-1} (out of {len(data)} frames)"

    results = {
        'g2': g2_clean,
        'F': F,
        'lag_time_s': lag_time_s_clean,
        'identifier': identifier,
        'method': 'multitau',
        'start_frame': start_frame,
        'end_frame': end_frame,
        'num_points': len(lag_time_s_clean),
        'time_range': (lag_time_s_clean[0], lag_time_s_clean[-1]),
        'description': description
    }

    return results


In [ ]:
def calculate_lag_steps(num_levels, num_bufs, n_frames):
    """
    Calculate lag steps exactly as skbeam's multi_tau_auto_corr does.

    Parameters
    ----------
    num_levels : int
        Number of correlation levels
    num_bufs : int
        Number of buffers per level (must be even)
    n_frames : int
        Total number of frames in the movie

    Returns
    -------
    lag_steps : ndarray
        Array of lag steps (in units of frames)
    """
    if num_bufs % 2 != 0:
        raise ValueError("num_bufs must be even")

    lag_steps = []

    for level in range(num_levels):
        # Calculate how many times this level gets processed
        img_per_level = n_frames // (2 ** level)

        # Level 0: all buffers (i=0 to num_bufs-1)
        # Higher levels: only second half (i=num_bufs//2 to num_bufs-1)
        i_min = 0 if level == 0 else num_bufs // 2

        # Loop constrained by both num_bufs and img_per_level
        for i in range(i_min, min(img_per_level, num_bufs)):
            lag = i * (2 ** level)
            lag_steps.append(lag)

    # Remove duplicates and sort
    lag_steps = np.unique(lag_steps)

    return lag_steps

In [ ]:
lag_steps = calculate_lag_steps(num_levels=12, num_bufs=8, n_frames=1020)

print(f"Lag steps: {lag_steps}")
print(f"\nTotal lag points: {len(lag_steps)}")
print(f"Max lag: {lag_steps[-1]} frames")


### Functions for fitting

Extract dynamics parameters from correlation functions using stretched exponential models.

`stretched_exponential`: Stretched exponential decay model function for fitting XPCS g2(tau) correlation data.

`fit_F_corr`: Pure fitting function that returns comprehensive fit results without plotting for flexible reuse.

`plot_fit_results`: Create diagnostic plots with fit curves and residuals from fit_F_corr results.

In [ ]:
def stretched_exponential(lag_time_s, A, Gamma, B, beta):
    """
    Stretched exponential function for XPCS g2 correlation analysis.

    This function implements the stretched exponential decay model commonly
    used to fit XPCS correlation data: g2(tau) = A * exp(-(Gamma * tau)^beta) + B

    Parameters:
    -----------
    lag_time_s : array
        Time lag values in seconds
    A : float
        Amplitude/contrast parameter
    Gamma : float
        Characteristic relaxation rate (1/tau_characteristic)
    B : float
        Baseline offset
    beta : float
        Stretching exponent (beta=1: pure exponential, beta<1: sub-exponential)

    Returns:
    --------
    array
        g2(tau) values calculated from the stretched exponential model
    """
    return A * np.exp(-(Gamma * lag_time_s) ** beta) + B


In [ ]:
def fit_F_corr(F_data, lag_time_s, roi_name, start_idx=0, end_idx=None, initial_gamma=1e-3,
               print_results=True):
    """
    Fit F(tau) correlation data with stretched exponential model.

    This function performs pure fitting without plotting, allowing flexible
    reuse of fit results for different visualization or analysis purposes.

    Parameters:
    -----------
    F_data : array
        F(tau) correlation data to fit
    lag_time_s : array
        Time lag array in seconds
    roi_name : str
        ROI identifier for labeling and error reporting
    start_idx : int, optional
        Starting index for fitting range (default: 0)
    end_idx : int, optional
        Ending index for fitting range (None uses all data)
    initial_gamma : float, optional
        Initial guess for Gamma parameter (default: 1e-3)
    print_results : bool, optional
        Print final fit outcome (default: True)

    Returns:
    --------
    dict
        Comprehensive fit results containing:
        - 'success': boolean indicating if fit succeeded
        - 'A', 'Gamma', 'B', 'beta': fitted parameters
        - 'tau_char': characteristic time (1/Gamma)
        - 'r_squared': coefficient of determination
        - 'param_bounds': dictionary with min/max bounds for each parameter
        - 'fit_result': complete lmfit result object
        - 'fit_times': time values used for fitting
        - 'fit_data': F data values used for fitting
        - 'fit_curve': fitted curve values
        - 'residuals': fit residuals
        - 'roi_name': ROI identifier
        - 'n_points_fitted': number of data points used
    """

    # Data selection and preparation
    if end_idx is None:
        end_idx = len(F_data)
    elif end_idx < 0:
        end_idx = len(F_data) + end_idx

    # Prepare fitting data
    fit_times = lag_time_s[start_idx:end_idx]
    fit_data = F_data[start_idx:end_idx].flatten()  # Ensure 1D

    # Always print progress
    print(f"Fitting {roi_name}... Using {len(fit_data)} points (idx {start_idx}:{end_idx})")

    # Data-driven parameter initialization
    F_start = np.max(fit_data)
    F_end = np.min(fit_data)
    decay_amplitude = F_start - F_end

    # Set up model with data-driven bounds
    model = Model(stretched_exponential)
    params = model.make_params(
        A={'value': decay_amplitude, 'min': decay_amplitude*0.1, 'max': decay_amplitude*5},
        Gamma={'value': initial_gamma, 'min': 1e-6, 'max': 1e-1},
        B={'value': F_end, 'min': F_end*0.995, 'max': F_start*1.005},
        beta={'value': 1.0, 'min': 0.1, 'max': 3.0}
    )

    # Store parameter bounds for later use
    param_bounds = {
        'A': {'min': params['A'].min, 'max': params['A'].max},
        'B': {'min': params['B'].min, 'max': params['B'].max},
        'beta': {'min': params['beta'].min, 'max': params['beta'].max},
        'Gamma': {'min': params['Gamma'].min, 'max': params['Gamma'].max}
    }

    try:
        # Perform fitting
        fit_result = model.fit(fit_data, params, lag_time_s=fit_times)

        if fit_result.success:
            # Extract fitted parameters
            A_fit = fit_result.best_values['A']
            Gamma_fit = fit_result.best_values['Gamma']
            B_fit = fit_result.best_values['B']
            beta_fit = fit_result.best_values['beta']
            tau_char = 1 / Gamma_fit
            r_squared = 1 - fit_result.redchi / np.var(fit_data)

            # Check for boundary hits (always warn)
            boundary_warnings = []
            tolerance = 1e-10

            if abs(A_fit - params['A'].max) < tolerance:
                boundary_warnings.append("A(max)")
            elif abs(A_fit - params['A'].min) < tolerance:
                boundary_warnings.append("A(min)")

            if abs(Gamma_fit - params['Gamma'].max) < tolerance:
                boundary_warnings.append("Gamma(max)")
            elif abs(Gamma_fit - params['Gamma'].min) < tolerance:
                boundary_warnings.append("Gamma(min)")

            if abs(B_fit - params['B'].max) < tolerance:
                boundary_warnings.append("B(max)")
            elif abs(B_fit - params['B'].min) < tolerance:
                boundary_warnings.append("B(min)")

            if abs(beta_fit - params['beta'].max) < tolerance:
                boundary_warnings.append("beta(max)")
            elif abs(beta_fit - params['beta'].min) < tolerance:
                boundary_warnings.append("beta(min)")

            if boundary_warnings:
                print(f"⚠️  Parameter(s) hit boundary: {', '.join(boundary_warnings)} - consider relaxing bounds")

            if print_results:
                print(f"✅ Fit successful! A = {A_fit:.6f}, B = {B_fit:.6f}, tau_c = {tau_char:.1f}s, beta = {beta_fit:.3f}, R^2 = {r_squared:.4f}")

            return {
                'success': True,
                'roi_name': roi_name,
                'A': A_fit,
                'Gamma': Gamma_fit,
                'B': B_fit,
                'beta': beta_fit,
                'tau_char': tau_char,
                'r_squared': r_squared,
                'param_bounds': param_bounds,
                'fit_result': fit_result,
                'fit_times': fit_times,
                'fit_data': fit_data,
                'fit_curve': fit_result.best_fit,
                'residuals': fit_data - fit_result.best_fit,
                'n_points_fitted': len(fit_data)
            }
        else:
            if print_results:
                print(f"❌ Fit failed: {fit_result.message}")
            return {
                'success': False,
                'roi_name': roi_name,
                'message': fit_result.message,
                'param_bounds': param_bounds,
                'fit_result': fit_result
            }

    except Exception as e:
        if print_results:
            print(f"❌ Fitting error: {str(e)}")
        return {
            'success': False,
            'roi_name': roi_name,
            'error': str(e),
            'param_bounds': param_bounds
        }


In [ ]:
def plot_fit_results(fit_result, F_data, lag_time_s, roi_name, figsize=(12, 5), print_summary=True):
    """
    Create diagnostic plots for fitted F(tau) correlation data.

    This function generates standardized plots showing the fit quality,
    residuals, and key parameters for correlation function analysis.

    Parameters:
    -----------
    fit_result : dict
        Output from fit_F_corr() containing fit parameters and data
    F_data : array
        Complete F(tau) correlation data (for context)
    lag_time_s : array
        Complete time lag array in seconds (for context)
    roi_name : str
        ROI identifier for plot titles
    figsize : tuple, optional
        Figure size (width, height) for the plots
    print_summary : bool, optional
        Print detailed fit summary with parameter bounds (default: True)
    """

    if not fit_result['success']:
        print(f"❌ Cannot plot - {roi_name} fit failed: {fit_result.get('message', 'Unknown error')}")
        return

    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

    # Left plot: Data and fit
    # Plot all data points (context)
    ax1.semilogx(lag_time_s, F_data, 'o', color='lightgray', ms=2,
                alpha=0.5, label='All data', zorder=1)

    # Plot fitted data points (prominent)
    ax1.semilogx(fit_result['fit_times'], fit_result['fit_data'], 'ko', ms=3,
                label=f'{roi_name} fitted data', zorder=3)

    # Plot fit curve
    ax1.semilogx(fit_result['fit_times'], fit_result['fit_curve'], 'r-', linewidth=2.5,
                label='Fit', zorder=2)

    ax1.set_xlabel('lag_time_s (s)', fontsize=12)
    ax1.set_ylabel('F(tau)', fontsize=12)
    ax1.set_title(f'{roi_name}: tau_c = {fit_result["tau_char"]:.1f}s, beta = {fit_result["beta"]:.3f}')
    ax1.grid(True, alpha=0.3, which='major')
    ax1.grid(True, alpha=0.15, which='minor')
    ax1.legend()

    # Right plot: Residuals
    ax2.semilogx(fit_result['fit_times'], fit_result['residuals'], 'go', ms=2.5, alpha=0.7)
    ax2.axhline(y=0, color='black', linestyle='-', alpha=0.6, linewidth=1)
    ax2.set_xlabel('lag_time_s (s)', fontsize=12)
    ax2.set_ylabel('Residuals', fontsize=12)
    ax2.set_title(f'{roi_name} Fit Residuals (R^2 = {fit_result["r_squared"]:.4f})')
    ax2.grid(True, alpha=0.3, which='major')
    ax2.grid(True, alpha=0.15, which='minor')

    plt.tight_layout()
    plt.show()

    # Print detailed summary with parameter bounds
    if print_summary:
        bounds = fit_result['param_bounds']
        print(f"\n✅ Fitting successful (R^2 = {fit_result['r_squared']:.4f})! Results:")
        print(f"    A = {fit_result['A']:.6f} (min: {bounds['A']['min']:.6f}, max: {bounds['A']['max']:.6f})")
        print(f"    B = {fit_result['B']:.6f} (min: {bounds['B']['min']:.6f}, max: {bounds['B']['max']:.6f})")
        print(f"    beta = {fit_result['beta']:.3f} (min: {bounds['beta']['min']:.3f}, max: {bounds['beta']['max']:.3f})")
        print(f"    Gamma = {fit_result['Gamma']:.3e} (min: {bounds['Gamma']['min']:.2e}, max: {bounds['Gamma']['max']:.2e}) 1/s")
        print(f"    tau_c = {fit_result['tau_char']:.1f}s")
        print(f"    Points fitted: {fit_result['n_points_fitted']}\n")


## Locate Datafiles

The experimental data and scripts are stored in the Frano X-lab's Google Drive, under _Beamtimes_.


In [ ]:
# data_path is the "Data" folder that contains all data and scripts, stored in the X Lab shared drive.
data_path = r'/content/drive/MyDrive/2026-02-24_ALS_COSMIC_XPCS_FNS_with_Shan/Data/'

# Check if path exists
if not os.path.exists(data_path):
    print(f"❌ Path not found: {data_path}")
    print("Please check if Google Drive is mounted and path is correct.")
else:
    # Get all files and sort by modification time
    files = sorted(glob(os.path.join(data_path, '*')), key=os.path.getmtime)

    print(f"Data directory: {data_path}")
    print(f"Total files found: {len(files)}")
    print("-" * 60)

    for i, f in enumerate(files):
        print(f"{i:2d} {basename(f)}")

In [ ]:
# Initialize XPCS Data Manager

print("Setting up XPCS Data Manager ...\n")
results_path = os.path.join(data_path, 'Francis_XPCS', 'Results_2')

# Create results directory if it doesn't exist
os.makedirs(results_path, exist_ok=True)

# Initialize the new data manager
dm = setup_xpcs_manager(storage_path=results_path)

print()
dm.get_summary()

# ONE-TEMPERATURE ANALYSIS

**Pipeline overview:**

1. Data loading, with quality assessment
2. Data preparation: ROI definition, waterfall analysis, normalization, and peak calculation
3. Correlation calculation: Two-time correlation, one-time correlation, and  dynamics extraction.
4. Results storing in the data manager.

## Load and Display Data

Import selected HDF5 datasets with comprehensive metadata extraction and initial interactive visualization.


In [ ]:
file_index = 66  # Change to desired file index from the inventory above # EDIT HERE FOR INDEX. 7/27

# Validate file index
if file_index >= len(files):
    print(f"❌ Invalid file_index {file_index}. Available range: 0-{len(files)-1}")
else:
    file = files[file_index]
    print(f"Loading: {basename(file)}")
    print()

# Open HDF5 file and explore structure
with h5py.File(file, 'r') as f:
    print("HDF5 file structure:")
    print("-" * 60)
    print(f"Root keys: {list(f.keys())}")
    print(f"Entry1 keys: {list(f['entry1'].keys())}")
    print(f"Instrument keys: {list(f['entry1']['instrument_1'].keys())}")
    print(f"Detector keys: {list(f['entry1']['instrument_1']['detector_1'].keys())}")
    print(f"LabView data keys: {list(f['entry1']['instrument_1']['labview_data'].keys())}")

    # Load detector data
    data = np.squeeze(f['entry1']['instrument_1']['detector_1']['data'][:]).astype(np.float32) # July

    # Sample parameters
    sample_theta = f['entry1']['instrument_1']['labview_data']['sample_rotate_steppertheta'][:]
    sample_translation = f['entry1']['instrument_1']['labview_data']['sample_translate'][:]
    sample_lift = f['entry1']['instrument_1']['labview_data']['sample_lift'][:]

    # Beamline and endstation parameters
    beamline_energy = f['entry1']['instrument_1']['labview_data']['beamline_energy'][:]
    temperature = f['entry1']['instrument_1']['labview_data']['LS_LLHTA'][:]
    i0_left = f['entry1']['instrument_1']['labview_data']['XS111LeftBladecurrent_diode'][:]
    i0_right = f['entry1']['instrument_1']['labview_data']['XS111RightBladecurrent_diode'][:]
    polarization = f['entry1']['instrument_1']['labview_data']['EPU_Polarization'][:]

    # Detector configuration
    exposure_time = f['entry1']['instrument_1']['detector_1']['count_time'][()]
    readout_time = f['entry1']['instrument_1']['detector_1']['detector_readout_time'][()]
    sample_detector_distance = f['entry1']['instrument_1']['detector_1']['distance'][()]

# Display experimental parameters
print("\nExperimental parameters")
print("-" * 60)
print(f"Data shape: {data.shape} (frames, height, width)")
print(f"Exposure time: {exposure_time:.1f} ms")
print(f"Readout time: {readout_time:.1f} ms")
print(f"Sample-detector distance: {sample_detector_distance:.3f} m")
print(f"Total measurement time: {len(data) * (exposure_time + readout_time) / 1000:.1f} s")

print(f"Temperature range: {np.min(temperature):.1f} - {np.max(temperature):.2f} K")
print(f"Temperature stability (std): ±{np.std(temperature):.3f} K")
print(f"Beamline energy: {np.mean(beamline_energy):.1f} eV")

print()
print(f"✅ Data loading complete: {len(data)} frames @ {np.mean(temperature):.1f}K")

In [ ]:
print(polarization)

In [ ]:
# Display the detector image using interactive frame viewer
print("INTERACTIVE DETECTOR IMAGE VIEWER")
print("-" * 60 + "\n")
interactive_frame_viewer(data, vmin=0, vmax=1000)
# interactive_frame_viewer(data[:,0:150,200:300], vmin=0, vmax=1000)    # Zoom in to the peak

In [ ]:
# Beam intensity stability monitoring
print("BEAM STABILITY ANALYSIS")
print("-" * 60)

# Calculate mean intensity per frame
frame_intensities = np.mean(data, axis=(1, 2))

# Create stability plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Intensity stability plot
frame_numbers = np.arange(len(data))
ax1.plot(frame_numbers, frame_intensities)
ax1.set_xlabel('Frame Number')
ax1.set_ylabel('Mean Frame Intensity (counts)')
ax1.set_title('Beam Intensity Stability')
ax1.grid(True)

# Temperature stability plot
ax2.plot(temperature)
ax2.set_xlabel('Frame Number')
ax2.set_ylabel('Temperature (K)')
ax2.set_title('Sample Temperature Stability')
ax2.grid(True)

plt.tight_layout()
plt.show()


## Define ROIs

Define regions of interest for signal extraction with comprehensive boundary validation and peak identification. ROI selection focuses on coherent scattering signal while avoiding detector artifacts and low-intensity regions.


In [ ]:
# Define Main and Background ROI
print("ROI DEFINITION 1")
print("-" * 60)

# ROI 0 - Main signal ROI for general correlation analysis
roi_0_x, roi_0_y = 220, 0   #242, 0
roi_0_width, roi_0_height = 60, 90
roi_0 = create_roi_slice(roi_0_x, roi_0_y, roi_0_width, roi_0_height, data.shape, roi_name="ROI 0")

# Background ROI - for noise characterization (optional)
bg_x, bg_y = 0, 300
bg_width, bg_height = 80, 80
roi_bg = create_roi_slice(bg_x, bg_y, bg_width, bg_height, data.shape, roi_name="ROI BG")

# Extract ROI data
roi_0_data = data[(..., *roi_0)]
roi_bg_data = data[(..., *roi_bg)]
print(f"Generated: roi_0_data {roi_0_data.shape}, roi_bg_data {roi_bg_data.shape}")

# Find peak position within ROI 0
roi_0_mean = np.mean(roi_0_data, axis=0)
peak_idx = np.unravel_index(np.argmax(roi_0_mean), roi_0_mean.shape)
peak_y_roi, peak_x_roi = peak_idx
peak_x_detector = roi_0_x + peak_x_roi  # Convert to detector coordinates
peak_y_detector = roi_0_y + peak_y_roi

print(f"Peak at detector coordinates: ({peak_x_detector}, {peak_y_detector})\n")

# Display ROIs on mean detector image
detector_mean = np.mean(data, axis=0)  # Use mean instead of data[0]
roi_0_mean_image = np.mean(roi_0_data, axis=0)  # Use mean instead of roi_0_data[0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Full detector and roi_0 visualization
im1 = ax1.imshow(detector_mean, cmap='viridis', vmin=0, vmax=1200)
ax1.set_xlabel('Pixel X')
ax1.set_ylabel('Pixel Y')
ax1.set_title('ROI_0 on Full Detector')
plt.colorbar(im1, ax=ax1)

# Add ROI rectangles
roi_0_rect = patches.Rectangle((roi_0_x, roi_0_y), roi_0_width, roi_0_height,
                               linewidth=2, edgecolor='red', facecolor='none', label='ROI 0')
bg_rect = patches.Rectangle((bg_x, bg_y), bg_width, bg_height,
                           linewidth=2, edgecolor='cyan', facecolor='none', label='Background ROI')
ax1.scatter(peak_x_detector, peak_y_detector, c='red', s=100, marker='x', linewidth=2, label='Peak')
ax1.add_patch(roi_0_rect)
ax1.add_patch(bg_rect)
ax1.legend()

# Show ROI 0 detail
im2 = ax2.imshow(roi_0_mean_image, cmap='viridis', vmin=0, vmax=1200)
ax2.set_xlabel('Pixel X (ROI)')
ax2.set_ylabel('Pixel Y (ROI)')
ax2.set_title(f'ROI 0 ({roi_0_width}×{roi_0_height} pixels)')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
# filename = f'{np.mean(temperature):.0f}K_ROI_details_1.png'
# plt.savefig(os.path.join(results_path, filename), dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Interactive ROI viewer
print("INTERACTIVE ROI VIEWERS 1")
print("-" * 60)
interactive_frame_viewer(roi_0_data, vmin=0, vmax=1500, figsize=(7, 5), set_title='ROI 0')
# interactive_frame_viewer(roi_bg_data, vmin=0, vmax=500, figsize=(6, 5), set_title='ROI bg')

In [ ]:
# Define finer ROIs
print("ROI DEFINITION 2")
print("-" * 60)

# # ROI 1 - Smaller ROI around peak (~20×35 pixels) Tempfix
# roi_1_x = peak_x_detector - 3   # -16
# roi_1_y = peak_y_detector  # CHANGED, it produced a negative error
# roi_1_width, roi_1_height = 20, 35
# roi_1 = create_roi_slice(roi_1_x, roi_1_y, roi_1_width, roi_1_height, data.shape, roi_name="ROI 1") analysis_data.shape

if peak_x_detector - 3 < 0:
    roi_1_x = 0
elif peak_x_detector + 20 > data.shape[2]:
    roi_1_x = data.shape[2] - 20
else:
  roi_1_x = peak_x_detector - 3   # -16

if peak_y_detector - 1 < 0:
  roi_1_y = 0
elif peak_y_detector + 35 > data.shape[1]:
  roi_1_y = data.shape[1] - 35
else:
  roi_1_y = peak_y_detector -1
roi_1_width, roi_1_height = 20, 35
roi_1 = create_roi_slice(roi_1_x, roi_1_y, roi_1_width, roi_1_height, data.shape, roi_name="ROI 1")

# ROI 2 - Small ROI centered on peak with edge handling
roi_2_width, roi_2_height = 8, 3
roi_2_x = max(0, peak_x_detector - roi_2_width//2)  # Center on peak with boundary check
roi_2_y = max(0, peak_y_detector - roi_2_height//2)
roi_2 = create_roi_slice(roi_2_x, roi_2_y, roi_2_width, roi_2_height, data.shape, roi_name="ROI 2")

# ROI 3 - Smaller ROI than ROI 1
roi_3_width, roi_3_height = 10, 12
roi_3_x = roi_1_x + 4
roi_3_y = roi_1_y + 22
roi_3 = create_roi_slice(roi_3_x, roi_3_y, roi_3_width, roi_3_height, data.shape, roi_name="ROI 3")

# Extract ROI data
roi_1_data = data[(..., *roi_1)]
roi_2_data = data[(..., *roi_2)]
roi_3_data = data[(..., *roi_3)]

print(f"Generated: roi_1_data {roi_1_data.shape}, roi_2_data {roi_2_data.shape}, roi_3_data {roi_3_data.shape}\n")

# Create mean images for visualization
roi_0_mean_image = np.mean(roi_0_data, axis=0)
roi_1_mean_image = np.mean(roi_1_data, axis=0)
roi_2_mean_image = np.mean(roi_2_data, axis=0)
roi_3_mean_image = np.mean(roi_3_data, axis=0)

# Create 2×2 plot layout
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: ROI 0 with smaller ROIs marked
im1 = ax1.imshow(roi_0_mean_image, cmap='viridis', vmin=0, vmax=2000)
ax1.set_xlabel('Pixel X (ROI 0)')
ax1.set_ylabel('Pixel Y (ROI 0)')
ax1.set_title(f'ROI 0 with Smaller ROIs')
plt.colorbar(im1, ax=ax1)

# Add ROI 1 and ROI 2 rectangles (coordinates relative to ROI 0)
roi_1_rect = patches.Rectangle((roi_1_x-roi_0_x, roi_1_y-roi_0_y), roi_1_width, roi_1_height,
                               linewidth=2, edgecolor='tab:orange', facecolor='none', label='ROI 1')
roi_2_rect = patches.Rectangle((roi_2_x-roi_0_x, roi_2_y-roi_0_y), roi_2_width, roi_2_height,
                               linewidth=2, edgecolor='yellow', facecolor='none', label='ROI 2')
ax1.add_patch(roi_1_rect)
ax1.add_patch(roi_2_rect)
ax1.scatter(peak_x_roi, peak_y_roi, c='red', s=100, marker='x', linewidth=2, label='Peak')
ax1.legend()

# Plot 2: ROI 1 detail
im2 = ax2.imshow(roi_1_mean_image, cmap='viridis', vmin=0, vmax=2000)
ax2.set_xlabel('Pixel X (ROI 1)')
ax2.set_ylabel('Pixel Y (ROI 1)')
ax2.set_title(f'ROI 1 ({roi_1_width}×{roi_1_height} pixels)')
plt.colorbar(im2, ax=ax2)

# Add ROI 3 patch within ROI 1 (coordinates relative to ROI 1)
roi_3_rect_in_roi1 = patches.Rectangle((roi_3_x-roi_1_x, roi_3_y-roi_1_y), roi_3_width, roi_3_height,
                                       linewidth=2, edgecolor='pink', facecolor='none', label='ROI 3')
ax2.add_patch(roi_3_rect_in_roi1)
ax2.legend()

# Plot 3: ROI 2 detail
im3 = ax3.imshow(roi_2_mean_image, cmap='viridis', vmin=0, vmax=2500)
ax3.set_xlabel('Pixel X (ROI 2)')
ax3.set_ylabel('Pixel Y (ROI 2)')
ax3.set_title(f'ROI 2 ({roi_2_width}×{roi_2_height} pixels)')
plt.colorbar(im3, ax=ax3)

# Plot 4: ROI 3 detail
im4 = ax4.imshow(roi_3_mean_image, cmap='viridis', vmin=0, vmax=1500)
ax4.set_xlabel('Pixel X (ROI 3)')
ax4.set_ylabel('Pixel Y (ROI 3)')
ax4.set_title(f'ROI 3 ({roi_3_width}×{roi_3_height} pixels)')
plt.colorbar(im4, ax=ax4)

plt.tight_layout()
# filename = f'{np.mean(temperature):.1f}K_ROI_details_2.png'
# plt.savefig(os.path.join(results_path, filename), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print("INTERACTIVE ROI VIEWERS 2")
print("-" * 60)
# interactive_frame_viewer(roi_1_data, vmin=0, vmax=2000, figsize=(7, 5), set_title='ROI 1')
# interactive_frame_viewer(roi_2_data, vmin=0, vmax=2000, figsize=(6, 4), set_title='ROI 2')
interactive_frame_viewer(roi_3_data, vmin=0, vmax=1000, figsize=(6, 4), set_title='ROI 3')

## Normailization

In [ ]:
# ROI data extraction and normalization
print("NORMALIZATION")
print("-" * 60)

# Method 1: Background ROI normalization
print("Applying background ROI normalization...")
corrected_data = np.zeros_like(data)
for i in range(len(data)):
    corrected_data[i, :, :] = data[i, :, :] / np.mean(roi_bg_data[i, :, :])

# Method 2: Detector-wide mean normalization
# print("Applying detector-wide mean normalization...")
# corrected_data = np.zeros_like(data)
# for i in range(len(data)):
#     corrected_data[i, :, :] = data[i, :, :] / np.mean(data[i, :, :])

# Extract corrected ROI data for correlation analysis
smd_0 = corrected_data[(..., *roi_0)]
smd_1 = corrected_data[(..., *roi_1)]
smd_2 = corrected_data[(..., *roi_2)]
smd_3 = corrected_data[(..., *roi_3)]
smd_bg = corrected_data[(..., *roi_bg)]

# Diagnostic check - background ROI should have constant total intensity after normalization
bg_std = np.std(np.sum(smd_bg, axis=(1, 2)))
print(f"Normalization complete. Background ROI std: {bg_std:.6f} (should be ~0).")

print(f"\n📋 Data extraction summary")
print("-" * 60)
print(f"Corrected ROI 0 data (smd_0) shape: {smd_0.shape}")
print(f"Corrected ROI 1 data (smd_1) shape: {smd_1.shape}")
print(f"Corrected ROI 2 data (smd_2) shape: {smd_2.shape}")
print(f"Corrected ROI 3 data (smd_3) shape: {smd_3.shape}")
# print(f"Background ROI data (smd_bg) shape: {smd_bg.shape}")


In [ ]:
# Normalization effectiveness comparison
print("NORMALIZATION COMPARISON")
print("-" * 60)

# Calculate total intensities before and after normalization
intensities_before = [
    np.sum(roi_0_data, axis=(1, 2)),
    np.sum(roi_1_data, axis=(1, 2)),
    np.sum(roi_2_data, axis=(1, 2)),
    np.sum(roi_3_data, axis=(1, 2))
]

intensities_after = [
    np.sum(smd_0, axis=(1, 2)),
    np.sum(smd_1, axis=(1, 2)),
    np.sum(smd_2, axis=(1, 2)),
    np.sum(smd_3, axis=(1, 2))
]

roi_names = ['ROI 0', 'ROI 1', 'ROI 2', 'ROI 3']
frame_numbers = np.arange(len(data))

# Create 2×2 subplot layout with dual y-axes
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (before, after, name) in enumerate(zip(intensities_before, intensities_after, roi_names)):
    ax = axes[i]
    ax_twin = ax.twinx()

    # Plot before and after normalization
    ax.plot(frame_numbers, before, 'b-', linewidth=0.5, alpha=0.8)
    ax_twin.plot(frame_numbers, after, 'r-', linewidth=0.5, alpha=0.8)

    # Formatting
    ax.set_xlabel('Frame Number')
    ax.set_ylabel('Before (counts)', color='b')
    ax_twin.set_ylabel('After', color='r')
    ax.set_title(f'{name} Total Intensity')
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='y', labelcolor='b')
    ax_twin.tick_params(axis='y', labelcolor='r')

plt.tight_layout()
plt.show()

**Intensity Jump:** Background ROI normalization successfully reduced gradual beam fluctuations, but sudden intensity jumps remain and need segmented normalization.

In [ ]:
# Test different window sizes for jump correction THIS CODE OVER HERE is pretty unreusable. 7/27 - Found that it works for 66, but you must manually look at the graphs for jump analysis

window_sizes = [1, 3, 10, 25]#, 50]
# colors = ['gray','C0','C1']
colors = ['gray', 'blue', 'green', 'orange', 'red']

# Calculate rolling averages for total detector intensity
frame_numbers = np.arange(len(data))
detector_intensity = np.mean(data, axis=(1, 2))

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Full range view
for i, (window, color) in enumerate(zip(window_sizes, colors)):
    if window == 1:
        # No averaging - original data
        smoothed = detector_intensity
        label = f'Original (n={window})'
    else:
        # Rolling average
        smoothed = np.convolve(detector_intensity, np.ones(window)/window, mode='same')
        label = f'Avg (n={window})'

    ax1.plot(frame_numbers, smoothed, color=color, linewidth=1, alpha=0.8, label=label)

ax1.set_xlabel('Frame Number')
ax1.set_ylabel('Mean Detector Intensity')
ax1.set_title('Full Range: Rolling Average Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Zoom around first jump (frames 700-800)
zoom1_start, zoom1_end = 700, 800
for i, (window, color) in enumerate(zip(window_sizes, colors)):
    if window == 1:
        smoothed = detector_intensity
        label = f'Original (n={window})'
    else:
        smoothed = np.convolve(detector_intensity, np.ones(window)/window, mode='same')
        label = f'Avg (n={window})'

    ax2.plot(frame_numbers[zoom1_start:zoom1_end], smoothed[zoom1_start:zoom1_end],
             color=color, linewidth=2, alpha=0.8, label=label)

ax2.axvline(x=713, color='black', linestyle='--', alpha=0.5, label='Jump start')
ax2.axvline(x=719, color='black', linestyle='--', alpha=0.5, label='Jump end')
ax2.set_xlabel('Frame Number')
ax2.set_ylabel('Mean Detector Intensity')
ax2.set_title('First Jump Region (700-800)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Zoom around second jump (frames 1200-1350)
zoom2_start, zoom2_end = 1200, 1350
for i, (window, color) in enumerate(zip(window_sizes, colors)):
    if window == 1:
        smoothed = detector_intensity
        label = f'Original (n={window})'
    else:
        smoothed = np.convolve(detector_intensity, np.ones(window)/window, mode='same')
        label = f'Avg (n={window})'

    ax3.plot(frame_numbers[zoom2_start:zoom2_end], smoothed[zoom2_start:zoom2_end],
             color=color, linewidth=2, alpha=0.8, label=label)

ax3.axvline(x=1224, color='black', linestyle='--', alpha=0.5, label='Jump start')
ax3.axvline(x=1230, color='black', linestyle='--', alpha=0.5, label='Jump end')
ax3.set_xlabel('Frame Number')
ax3.set_ylabel('Mean Detector Intensity')
ax3.set_title('Second Jump Region (1200-1350)')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
## Linear fit test for Jump 1 CHANGED AGAIN 7/27
print("LINEAR FIT TEST - JUMP 1")
print("-" * 60)

# Calculate stable baselines using 10-frame averages
# before_jump1 = np.mean(data[729], axis=0)
# after_jump1 = np.mean(data[733],  axis=0)
# before_jump1 = np.mean(data[720:730], axis=0)
# after_jump1 = np.mean(data[733:743],  axis=0)
before_jump1 = np.mean(data[704:714], axis=(0,1))
after_jump1 = np.mean(data[718:728],  axis=(0,1)) # Notice here
# before_jump1 = data[729]
# after_jump1 = data[733]

# print(f"Before jump baseline: frames 720-729 (shape: {before_jump1.shape})")
# print(f"After jump baseline: frames 733-742 (shape: {after_jump1.shape})")

# Prepare data for linear fit
y_all = before_jump1.flatten()  # All pixels - intensity before jump
x_all = after_jump1.flatten()   # All pixels - intensity after jump

print(f"Total data points for fit: {len(x_all):,}")
print(f"Intensity range before jump: {x_all.min():.1f} - {x_all.max():.1f}")
print(f"Intensity range after jump: {y_all.min():.1f} - {y_all.max():.1f}")

# Linear fit using all pixels: y = A*x + B
A_fit = np.vstack([x_all, np.ones(len(x_all))]).T
coefficients, residuals, rank, s = np.linalg.lstsq(A_fit, y_all, rcond=None)
A, B = coefficients

# Calculate R-squared
y_pred = A * x_all + B
ss_res = np.sum((y_all - y_pred) ** 2)
ss_tot = np.sum((y_all - np.mean(y_all)) ** 2)
r_squared = 1 - (ss_res / ss_tot)

print(f"\nLinear fit results:")
print(f"  y = {A:.4f} * x + {B:.2f}")
print(f"  R² = {r_squared:.6f}")
print(f"  Residual sum of squares: {ss_res:.2e}")

# Create visualization with subsampled data
subsample_factor = 1  # Every 100th pixel for plotting
indices = np.arange(0, len(x_all), subsample_factor)
x_plot = x_all[indices]
y_plot = y_all[indices]

print(f"\nVisualization using {len(x_plot):,} subsampled points (every {subsample_factor}th pixel)")

# Create the plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# Left plot: Scatter plot with fit line
ax1.scatter(x_plot, y_plot, alpha=0.6, s=2, color='blue', label=f'Pixels (n={len(x_plot):,})')

# Plot fit line across full intensity range
x_line = np.linspace(x_all.min(), x_all.max(), 100)
y_line = A * x_line + B
ax1.plot(x_line, y_line, 'r-', linewidth=2, label=f'Fit: y = {A:.3f}x + {B:.1f}')

# Perfect correlation line for reference 7/27 LOOK HERE
ax1.plot(x_line, x_line, 'k--', alpha=0.5, linewidth=1, label='y = x (no change)')

ax1.set_xlabel('Intensity Before Jump')
ax1.set_ylabel('Intensity After Jump')
ax1.set_title(f'Jump 1 Linearity Test (R² = {r_squared:.4f})')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right plot: Residuals
residuals_plot = y_plot - (A * x_plot + B)
ax2.scatter(x_plot, residuals_plot, alpha=0.6, s=2, color='green')
ax2.axhline(y=0, color='red', linestyle='-', linewidth=1)
ax2.set_xlabel('Intensity Before Jump')
ax2.set_ylabel('Residuals (y - fit)')
ax2.set_title('Fit Residuals')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Assessment
print(f"\n📊 LINEARITY ASSESSMENT:")
if r_squared > 0.95:
    print(f"   ✅ Excellent linearity (R² = {r_squared:.4f}) - Global A,B correction is appropriate")
elif r_squared > 0.85:
    print(f"   ✅ Good linearity (R² = {r_squared:.4f}) - Global A,B correction should work well")
elif r_squared > 0.70:
    print(f"   ⚠️  Moderate linearity (R² = {r_squared:.4f}) - Global correction may work but check residuals")
else:
    print(f"   ❌ Poor linearity (R² = {r_squared:.4f}) - Consider per-pixel or non-linear correction")

print(f"   💡 Multiplier A = {A:.4f} ({'amplification' if A > 1 else 'attenuation'})")
print(f"   💡 Offset B = {B:.1f} ({'positive' if B > 0 else 'negative'} shift)")

In [ ]:
# Spatial analysis of Jump 1 - look for spatial patterns
print("SPATIAL JUMP ANALYSIS")
print("-" * 60)

# Calculate jump magnitude for each pixel
before_jump1 = np.mean(data[704:714], axis=0)  # 10 frames before jump
after_jump1 = np.mean(data[718:728], axis=0)   # 10 frames after jump

# Calculate different jump metrics
absolute_jump = after_jump1 - before_jump1                    # Absolute change
relative_jump = (after_jump1 - before_jump1) / before_jump1   # Relative change (%) 7/27 potential issue? Change from Text Message
ratio_jump = after_jump1 / before_jump1                       # Multiplicative factor

print(f"Jump magnitude statistics:")
print(f"  Absolute jump range: {absolute_jump.min():.1f} to {absolute_jump.max():.1f}")
print(f"  Relative jump range: {relative_jump.min():.1%} to {relative_jump.max():.1%}")
print(f"  Ratio range: {ratio_jump.min():.3f} to {ratio_jump.max():.3f}")

# Create spatial plots
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Absolute jump magnitude
im1 = ax1.imshow(absolute_jump, cmap='RdBu_r', origin='lower', vmin=-100, vmax=100)
ax1.set_xlabel('Pixel X')
ax1.set_ylabel('Pixel Y')
ax1.set_title('Absolute Jump (After - Before)')
plt.colorbar(im1, ax=ax1, label='Intensity Change')

# Add ROI overlays for reference
roi_0_rect = patches.Rectangle((roi_0_x, roi_0_y), roi_0_width, roi_0_height,
                               linewidth=2, edgecolor='yellow', facecolor='none', label='ROI 0')
ax1.add_patch(roi_0_rect)
ax1.legend()

# Plot 2: Relative jump magnitude
im2 = ax2.imshow(relative_jump, cmap='RdBu_r', origin='lower', vmin=-0.5, vmax=0.5)
ax2.set_xlabel('Pixel X')
ax2.set_ylabel('Pixel Y')
ax2.set_title('Relative Jump ((After - Before) / Before)')
plt.colorbar(im2, ax=ax2, label='Relative Change')

# Add ROI overlay
roi_0_rect2 = patches.Rectangle((roi_0_x, roi_0_y), roi_0_width, roi_0_height,
                                linewidth=2, edgecolor='yellow', facecolor='none', label='ROI 0')
ax2.add_patch(roi_0_rect2)
ax2.legend()

# Plot 3: Multiplicative ratio
im3 = ax3.imshow(ratio_jump, cmap='RdBu_r', origin='lower', vmin=0.5, vmax=1.5)
ax3.set_xlabel('Pixel X')
ax3.set_ylabel('Pixel Y')
ax3.set_title('Multiplicative Ratio (After / Before)')
plt.colorbar(im3, ax=ax3, label='Ratio')

# Add ROI overlay
roi_0_rect3 = patches.Rectangle((roi_0_x, roi_0_y), roi_0_width, roi_0_height,
                                linewidth=2, edgecolor='yellow', facecolor='none', label='ROI 0')
ax3.add_patch(roi_0_rect3)
ax3.legend()

# Plot 4: Cross-sections through the peak
# Take horizontal and vertical cuts through the peak
peak_x = roi_0_x + peak_x_roi
peak_y = roi_0_y + peak_y_roi

# Horizontal cut (constant y, varying x)
horizontal_cut_abs = absolute_jump[peak_y, :]
horizontal_cut_rel = relative_jump[peak_y, :]

# Vertical cut (constant x, varying y)
vertical_cut_abs = absolute_jump[:, peak_x]
vertical_cut_rel = relative_jump[:, peak_x]

ax4_twin = ax4.twinx()

# Plot cuts
x_pixels = np.arange(len(horizontal_cut_abs))
y_pixels = np.arange(len(vertical_cut_abs))

ax4.plot(x_pixels, horizontal_cut_abs, 'b-', linewidth=2, label='Horizontal cut (absolute)')
ax4.plot(y_pixels, vertical_cut_abs, 'g-', linewidth=2, label='Vertical cut (absolute)')
ax4_twin.plot(x_pixels, horizontal_cut_rel, 'b--', linewidth=2, alpha=0.7, label='Horizontal cut (relative)')
ax4_twin.plot(y_pixels, vertical_cut_rel, 'g--', linewidth=2, alpha=0.7, label='Vertical cut (relative)')

ax4.axvline(x=peak_x, color='red', linestyle=':', alpha=0.7, label='Peak X')
ax4.axvline(x=peak_y, color='orange', linestyle=':', alpha=0.7, label='Peak Y')

ax4.set_xlabel('Pixel Position')
ax4.set_ylabel('Absolute Jump', color='black')
ax4_twin.set_ylabel('Relative Jump', color='gray')
ax4.set_title(f'Cross-sections through Peak ({peak_x}, {peak_y})')
ax4.grid(True, alpha=0.3)
ax4.legend(loc='upper left')
ax4_twin.legend(loc='upper right')

plt.tight_layout()
plt.show()

# Analysis of spatial patterns
print(f"\n📊 SPATIAL PATTERN ANALYSIS:")

# Check for gradients
x_gradient = np.gradient(absolute_jump, axis=1)
y_gradient = np.gradient(absolute_jump, axis=0)
total_gradient = np.sqrt(x_gradient**2 + y_gradient**2)

print(f"  Max gradient magnitude: {np.max(total_gradient):.2f}")
print(f"  Mean gradient magnitude: {np.mean(total_gradient):.2f}")

# Check for uniformity in ROI region
roi_jump = absolute_jump[roi_0_y:roi_0_y+roi_0_height, roi_0_x:roi_0_x+roi_0_width]
roi_std = np.std(roi_jump)
roi_mean = np.mean(roi_jump)

print(f"  ROI jump uniformity: mean = {roi_mean:.2f}, std = {roi_std:.2f} (CV = {roi_std/roi_mean:.1%})")

if roi_std/roi_mean < 0.1:
    print(f"  ✅ ROI region shows relatively uniform jump behavior")
else:
    print(f"  ⚠️  ROI region shows significant spatial variation in jump")

# Check for detector-wide patterns
detector_std = np.std(absolute_jump)
detector_mean = np.mean(absolute_jump)
print(f"  Detector-wide variation: CV = {detector_std/detector_mean:.1%}")

if detector_std/detector_mean > 0.5:
    print(f"  🔍 Strong spatial patterns detected - jump is not uniform across detector")
else:

    print(f"  📍 Jump appears relatively uniform across detector")

In [ ]:
# 7/27 Not Accurate for Now. Not entirely sure what this does at the moment
frame_intensities_2 = np.mean(data[:,0:150,200:300], axis=(1, 2))
frame_intensities_1 = np.mean(data[:,150:300,0:100], axis=(1, 2))

plt.plot(frame_intensities   - np.mean(frame_intensities[10:20]))
plt.plot(frame_intensities_1 - np.mean(frame_intensities_1[10:20]))
plt.plot(frame_intensities_2 - np.mean(frame_intensities_2[10:20]))
plt.xlabel('Frame Number')
plt.ylabel('Mean Frame Intensity (counts)')
plt.xlim(840, 860)
plt.ylim(3, -8)
plt.title('Beam Intensity Stability')
plt.legend(['All', 'Low', 'High'])
plt.grid(True)
plt.show()

print(np.mean(frame_intensities[10:20])-np.mean(frame_intensities[30:40])    )
print(np.mean(frame_intensities_1[10:20])-np.mean(frame_intensities_1[30:40]))
print(np.mean(frame_intensities_2[10:20])-np.mean(frame_intensities_2[30:40]))
print()
print((np.mean(frame_intensities[10:20])-np.mean(frame_intensities[30:40])) / np.mean(frame_intensities[10:20]))
print((np.mean(frame_intensities_1[10:20])-np.mean(frame_intensities_1[30:40]))/ np.mean(frame_intensities_1[10:20]))
print((np.mean(frame_intensities_2[10:20])-np.mean(frame_intensities_2[30:40]))/ np.mean(frame_intensities_2[10:20]))

# print(np.mean(frame_intensities_2[600:700])-np.mean(frame_intensities_2[750:850]))
# print()
# print((np.mean(frame_intensities[600:700])-np.mean(frame_intensities[750:850])    ) / np.mean(frame_intensities[600:700]))
# print((np.mean(frame_intensities_1[600:700])-np.mean(frame_intensities_1[750:850])) / np.mean(frame_intensities_1[600:700]))
# print((np.mean(frame_intensities_2[600:700])-np.mean(frame_intensities_2[750:850])) / np.mean(frame_intensities_2[600:700]))

In [ ]:
# # CHANGING THIS
# frame_intensities_2 = np.mean(data[:,0:150,200:300], axis=(1, 2))
# frame_intensities_1 = np.mean(data[:,150:300,0:100], axis=(1, 2))

# plt.plot(frame_intensities   - np.mean(frame_intensities[750:850]))
# plt.plot(frame_intensities_1 - np.mean(frame_intensities_1[750:850]))
# plt.plot(frame_intensities_2 - np.mean(frame_intensities_2[750:850]))
# plt.xlabel('Frame Number')
# plt.ylabel('Mean Frame Intensity (counts)')
# plt.xlim(840, 860)
# plt.ylim(3, -8)
# plt.title('Beam Intensity Stability')
# plt.legend(['All', 'Low', 'High'])
# plt.grid(True)
# plt.show()

# print(np.mean(frame_intensities[1200:1260])-np.mean(frame_intensities[1290:1350])    )
# print(np.mean(frame_intensities_1[1200:1260])-np.mean(frame_intensities_1[1290:1350]))
# print(np.mean(frame_intensities_2[1200:1260])-np.mean(frame_intensities_2[1290:1350]))
# print()
# print((np.mean(frame_intensities[1200:1260])-np.mean(frame_intensities[1290:1350])) / np.mean(frame_intensities[1200:1260]))
# print((np.mean(frame_intensities_1[1200:1260])-np.mean(frame_intensities_1[1290:1350]))/ np.mean(frame_intensities_1[1200:1260]))
# print((np.mean(frame_intensities_2[1200:1260])-np.mean(frame_intensities_2[1290:1350]))/ np.mean(frame_intensities_2[1200:1260]))

# # print(np.mean(frame_intensities_2[600:700])-np.mean(frame_intensities_2[750:850]))
# # print()
# # print((np.mean(frame_intensities[600:700])-np.mean(frame_intensities[750:850])    ) / np.mean(frame_intensities[600:700]))
# # print((np.mean(frame_intensities_1[600:700])-np.mean(frame_intensities_1[750:850])) / np.mean(frame_intensities_1[600:700]))
# # print((np.mean(frame_intensities_2[600:700])-np.mean(frame_intensities_2[750:850])) / np.mean(frame_intensities_2[600:700]))

In [ ]:
# Jump correction - segmented normalization
print("JUMP CORRECTION")
print("-" * 40)

# Define segments excluding problematic transition frames
segments = [
    (0, 730),        # Segment 1: before first jump
    (733, 1276),     # Segment 2: between jumps
    (1280, len(data)) # Segment 3: after second jump
]

print(f"(Excluding problematic frames)")
print(f"Segment 1: frames {segments[0][0]}-{segments[0][1]-1}")
print(f"Segment 2: frames {segments[1][0]}-{segments[1][1]-1}")
print(f"Segment 3: frames {segments[2][0]}-{segments[2][1]-1}")

# Apply background ROI normalization to each segment separately
corrected_data_segmented = np.zeros_like(data)
bg_means_segmented = []

for i, (start, end) in enumerate(segments):
    print(f"\nProcessing Segment {i+1}")

    # Calculate background means for this segment only
    segment_bg_data = roi_bg_data[start:end]
    segment_bg_means = np.mean(segment_bg_data, axis=(1, 2))
    bg_means_segmented.append(segment_bg_means)

    # Apply normalization to this segment
    segment_data = data[start:end]
    corrected_data_segmented[start:end] = segment_data / segment_bg_means[:, np.newaxis, np.newaxis]

    print(f"  Background mean range: {segment_bg_means.min():.1f} - {segment_bg_means.max():.1f}")

# Calculate scaling factors to reconnect segments
# Using background ROI total intensity as reference
segment_bg_levels = []
for i, (start, end) in enumerate(segments):
    segment_bg_corrected = corrected_data_segmented[start:end][(..., *roi_bg)]
    segment_bg_level = np.mean(np.sum(segment_bg_corrected, axis=(1, 2)))
    segment_bg_levels.append(segment_bg_level)

print(f"\nBackground ROI levels after segmented normalization:")
for i, level in enumerate(segment_bg_levels):
    print(f"  Segment {i+1}: {level:.1f}")

# Apply scaling factors (scale to Segment 1 level)
reference_level = segment_bg_levels[0]
scaling_factors = [reference_level / level for level in segment_bg_levels]

print(f"\nScaling factors (relative to Segment 1):")
for i, factor in enumerate(scaling_factors):
    print(f"  Segment {i+1}: {factor:.4f}")

# Apply scaling to reconnect segments
for i, (start, end) in enumerate(segments):
    corrected_data_segmented[start:end] *= scaling_factors[i]

print(f"\nJump correction complete - segments normalized and reconnected")

# Replace the corrected_data with jump-corrected version
corrected_data = corrected_data_segmented.copy()

# Update ROI data with jump-corrected normalization
smd_0 = corrected_data[(..., *roi_0)]
smd_1 = corrected_data[(..., *roi_1)]
smd_2 = corrected_data[(..., *roi_2)]
smd_3 = corrected_data[(..., *roi_3)]
smd_bg = corrected_data[(..., *roi_bg)]

# Final diagnostic check
final_bg_std = np.std(np.sum(smd_bg, axis=(1, 2)))
print(f"Final background ROI std: {final_bg_std:.6f} (should be ~0)")

print(f"\n📋 Jump-corrected data ready for correlation analysis")

In [ ]:
# ROI data extraction and normalization - removes slow intensity drifts

print("Normalizing data to remove beam intensity fluctuations...")

corrected_data = np.zeros((data.shape))
for i in range(len(data)):
    corrected_data[i, :, :] = data[i, :, :] / np.mean(data[i, :, :])

# Normalize using background intensity
# corrected_data = np.zeros((data.shape))
# for i in range(len(data)):
#     corrected_data[i, :, :] = data[i, :, :] / np.mean(roi_bg_data[i, :, :])

# Extract corrected ROI data for correlation analysis
smd_0 = corrected_data[(..., *roi_0)]
smd_1 = corrected_data[(..., *roi_1)]
smd_2 = corrected_data[(..., *roi_2)]
smd_3 = corrected_data[(..., *roi_3)]
# smd_bg = corrected_data[(..., *roi_bg)]

print()
print(f"📋 ROI extraction summary")
print("-" * 60)
print(f"Corrected ROI 0 data (smd_0) shape: {smd_0.shape}")
print(f"Corrected ROI 1 data (smd_1) shape: {smd_1.shape}")
print(f"Corrected ROI 2 data (smd_2) shape: {smd_2.shape}")
print(f"Corrected ROI 3 data (smd_3) shape: {smd_3.shape}")
# print(f"Background ROI data (smd_bg) shape: {smd_bg.shape}")

## Waterfall plots

Waterfall plots reveal temporal dynamics by showing how intensity patterns evolve along specific cuts through the ROI over time. These plots help identify drift, aging, and non-equilibrium behavior.


In [ ]:
# Generate waterfall plots

print(f"Generating waterfall plots...")

print()
print(f"smd_0 shape: {smd_0.shape}")
print(f"smd_1 shape: {smd_1.shape}")
print(f"smd_3 shape: {smd_3.shape}")
print(f"Y-cut: intensity vs time at fixed x, shows vertical speckle deviation")
print(f"X-cut: intensity vs time at fixed y, shows horizontal speckle deviation")
print()

# ROI 0 waterfall plots
cut_Y_0 = smd_0[:, :, 40]  # Cut at pixel x=40 (Y direction data)
cut_X_0 = smd_0[:, 30, :]  # Cut at pixel y=40 (X direction data)

# ROI 1 waterfall plots
cut_Y_1 = smd_1[:, :, 8]   # Cut at pixel x=8 (Y direction data)
cut_X_1 = smd_1[:, 3, :]   # Cut at pixel y=3 (X direction data)

# ROI 3 waterfall plots
roi3_pixels = smd_3.reshape(smd_3.shape[0], -1)

# Alternative: Sum projection method (integrate intensity across one dimension)
# projdata_X_0 = np.zeros((smd_0.shape[0], smd_0.shape[2]))
# for i in range(smd_0.shape[0]):
#     projdata_X_0[i] = np.sum(smd_0[i], axis=0)

# projdata_Y_0 = np.zeros((smd_0.shape[0], smd_0.shape[1]))
# for i in range(smd_0.shape[0]):
#     projdata_Y_0[i] = np.sum(smd_0[i], axis=1)

# projdata_X_1 = np.zeros((smd_1.shape[0], smd_1.shape[2]))
# for i in range(smd_1.shape[0]):
#     projdata_X_1[i] = np.sum(smd_1[i], axis=0)

# projdata_Y_1 = np.zeros((smd_1.shape[0], smd_1.shape[1]))
# for i in range(smd_1.shape[0]):
#     projdata_Y_1[i] = np.sum(smd_1[i], axis=1)

# Display waterfall plots - ROI 0, ROI 1, and ROI 3
fig, ((ax1, ax2, ax5), (ax3, ax4, ax6)) = plt.subplots(2, 3, figsize=(18, 10))

# ROI 0 waterfalls
im1 = ax1.imshow(cut_Y_0, cmap='CMRmap', aspect='auto')
ax1.set_xlabel('Y Position (pixels)')
ax1.set_ylabel('Frame Number')
ax1.set_title('ROI 0: Cut along y direction')
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(cut_X_0, cmap='CMRmap', aspect='auto')
ax2.set_xlabel('X Position (pixels)')
ax2.set_ylabel('Frame Number')
ax2.set_title('ROI 0: Cut along x direction')
plt.colorbar(im2, ax=ax2)

# ROI 1 waterfalls
im3 = ax3.imshow(cut_Y_1, cmap='CMRmap', aspect='auto')
ax3.set_xlabel('Y Position (pixels)')
ax3.set_ylabel('Frame Number')
ax3.set_title('ROI 1: Cut along y direction')
plt.colorbar(im3, ax=ax3)

im4 = ax4.imshow(cut_X_1, cmap='CMRmap', aspect='auto')
ax4.set_xlabel('X Position (pixels)')
ax4.set_ylabel('Frame Number')
ax4.set_title('ROI 1: Cut along x direction')
plt.colorbar(im4, ax=ax4)

# ROI 3 waterfall - all pixels
im5 = ax5.imshow(roi3_pixels, cmap='CMRmap', aspect='auto')
ax5.set_xlabel('Frame Number')
ax5.set_ylabel('Pixel Number')
ax5.set_title(f'ROI 3: All {roi3_pixels.shape[1]} pixels vs time')
plt.colorbar(im5, ax=ax5)

# Empty subplot for symmetry
ax6.axis('off')

plt.tight_layout()
plt.show()

## Calculate Correlation Functions

Compute both two-time and one-time intensity correlation functions to characterize sample dynamics.
- Two-time correlations $g_2(t_1,t_2)$ reveal aging, drift, and non-equilibrium behavior by showing how correlations evolve with both observation time and lag time.
- One-time correlations $g_2(\tau)$ provide the traditional XPCS measurement of relaxation dynamics under equilibrium conditions.

## Mathematical Explanation of Correlation Functions and Fitting

This section details the mathematical underpinnings of the operations performed after "## Calculate Correlation Functions" in the notebook. XPCS (X-ray Photon Correlation Spectroscopy) relies heavily on these correlation functions to probe the dynamics of materials.

### 1. Two-Time Correlation Function: $g_2(t_1, t_2)$

The two-time correlation function $g_2(t_1, t_2)$ is a powerful tool to analyze non-stationary dynamics, aging, or intermittent processes in a sample. It quantifies the correlation between intensity fluctuations at two different absolute times, $t_1$ and $t_2$.

**Mathematical Definition:**

$$ g_2(t_1, t_2) = \frac{\langle I(t_1) I(t_2) \rangle}{\langle I(t_1) \rangle \langle I(t_2) \rangle} $$

Where:
- $I(t)$ is the scattered intensity from a region of interest (ROI) at time $t$.
- $\langle ... \rangle$ denotes an ensemble average (in this context, often an average over pixels within an ROI).

**Interpretation:**
- If the dynamics are stationary (i.e., time-invariant), $g_2(t_1, t_2)$ depends only on the time difference $\tau = |t_2 - t_1|$, reducing to the one-time correlation function $g_2(\tau)$.
- Deviations from this diagonal behavior ($t_1 \neq t_2$) indicate non-stationary behavior, such as aging (where the dynamics slow down over time), drift, or intermittent events.

**In the Notebook:**
- The function `skbeam.core.correlation.two_time_corr` is used to compute this, taking the `labeled_array` (which defines the ROI pixels) and the `analysis_data` (intensity stack) as inputs. The `num_levels` and `num_bufs` parameters control the algorithm's efficiency, similar to the multi-tau method for one-time correlation.
- The output `g2_two_time` is a 2D matrix where each element $(i, j)$ corresponds to $g_2(t_i, t_j)$.
- **Aging analysis** looks at the diagonal elements $g_2(t,t)$, which should be constant for a stationary system. A changing diagonal indicates aging.
- **Off-diagonal analysis** examines slices parallel to the main diagonal to check for stationarity at different lag times.

### 2. One-Time Correlation Function: $g_2(\tau)$ and $F(\tau)$

For systems with stationary dynamics, the correlation function depends only on the time difference, $\tau$, between two measurements. This is known as the one-time correlation function.

$$ g_2(\tau) = \frac{\langle I(t) I(t + \tau) \rangle}{\langle I(t) \rangle \langle I(t + \tau) \rangle} $$

In XPCS, the dynamics are often characterized by the **Intermediate Scattering Function (ISF)**, $F(q, \tau)$, which is related to $g_2(\tau)$:

$$ g_2(\tau) = 1 + \beta |F(q, \tau)|^2 $$

Where $\beta$ is the speckle contrast, which depends on experimental conditions and sample properties. Often, $F(\tau)$ is extracted directly from $g_2(\tau)$ by assuming a constant $\beta$ (e.g., using the average plateau of $g_2$ at short lag times).

**In the Notebook, two primary methods are used for computing $g_2(\tau)$:**

#### a. FFT Method (`calculate_corr_fft` and `g2_scipy_fft`)

This method leverages the Wiener-Khinchin theorem, which states that the autocorrelation function of a signal is the inverse Fourier transform of its power spectral density. For discrete signals, this translates to:

$$ C(\tau) = \text{IFFT} [ |\text{FFT}(I(t))|^2 ] $$

Where $C(\tau)$ is the unnormalized correlation. After normalization, we get $g_2(\tau)$.

**Advantages:**
- **Computational Efficiency:** For a dataset of $N$ frames, the direct computation of $g_2(\tau)$ scales as $O(N^2)$, while the FFT method scales as $O(N \log N)$, making it much faster for large datasets.
- **Uniform Time Lags:** It naturally produces correlation values for all integer lag times with uniform spacing.

**Mathematical Steps in `g2_scipy_fft`:**
1.  **Zero-Padding:** The intensity stack `img_stack` is zero-padded to prevent circular correlation artifacts. This effectively makes the length of the signal larger for FFT computation.
2.  **Forward FFT:** The `scipy.fft.fft` function computes the Fast Fourier Transform of the padded intensity data.
3.  **Power Spectral Density (Implicit):** The product `img_stack_fft * img_stack_fft.conj()` effectively computes the magnitude squared of the FFT, which is related to the power spectral density.
4.  **Inverse FFT:** `scipy.fft.ifft` transforms this back to the time domain, yielding the unnormalized correlation `numerator_base`.
5.  **Normalization:** The `numerator_base` is then divided by the number of contributing pairs at each lag time (`n_elements`) and by the square of the temporal average of intensities (`denominator_base`) to obtain the normalized $g_2$ function.
6.  **Ensemble Average:** Finally, `numerator_base.mean(axis=(1,2))` and `denominator_base.mean(axis=(1,2))` compute the average over all pixels in the ROI.

**Conversion to $F(\tau)$:**
- The `calculate_corr_fft` function then calculates $F(\tau)$ using the formula: $F(\tau) = \sqrt{\frac{g_2(\tau) - 1}{\langle g_2(\tau_{short}) \rangle - 1}}$, where $\langle g_2(\tau_{short}) \rangle$ is the average of $g_2$ at very short lag times, approximating $1 + \beta$. This effectively removes the baseline and normalizes the amplitude to range from 0 to 1 for the ISF.

#### b. Multi-tau Method (`calculate_corr_multitau`)

The multi-tau algorithm is a hierarchical averaging scheme that provides logarithmically spaced time lags. It's particularly useful for probing dynamics that span many decades in time.

**How it Works (Conceptually):**
1.  **Short Lag Times:** For very short lag times, every frame is used to compute the correlation, similar to a direct method. This provides high resolution at short $\tau$.
2.  **Longer Lag Times:** For longer lag times, frames are successively averaged together (e.g., pairs of frames are averaged, then pairs of these averaged frames, and so on). The correlation is then computed using these averaged frames.
3.  **Logarithmic Spacing:** This hierarchical averaging naturally leads to logarithmically spaced lag times, allowing access to very long $\tau$ values without excessive computation or storage.

**Advantages:**
- **Wide Time Range:** Efficiently covers a broad range of time scales, from nanoseconds to seconds or even hours.
- **Reduced Data Size:** Significantly reduces the amount of data processed for long lag times.
- **Improved Statistics:** By averaging frames for longer lag times, it can provide better statistics for slow dynamics.

**In the Notebook (`corr.multi_tau_auto_corr` from `skbeam`):**
- `num_levels`: Controls the number of averaging levels, determining the maximum lag time accessible. Higher `num_levels` extend the observable time range.
- `num_bufs`: Dictates how many buffers (individual data points) are kept at each level before averaging. More `num_bufs` generally lead to better statistics for each correlated point.
- The `calculate_lag_steps` function demonstrates how `lag_steps` are generated, showing the logarithmic progression.

**Conversion to $F(\tau)$:**
- Similar to the FFT method, $F(\tau)$ is derived from $g_2(\tau)$ using the formula: $F(\tau) = \sqrt{\frac{g_2(\tau) - 1}{\langle g_2(\tau_{short}) \rangle - 1}}$, effectively normalizing the ISF.

### 3. Stretched Exponential Fitting (Kohlrausch-Williams-Watts or KWW Model)

After calculating the intermediate scattering function $F(\tau)$, the next step is often to fit it to a theoretical model to extract characteristic parameters of the dynamics. A common model in XPCS, especially for complex or heterogeneous dynamics, is the **stretched exponential** (Kohlrausch-Williams-Watts or KWW) function.

**Mathematical Form (`stretched_exponential` function):**

The full $g_2(\tau)$ is typically fit to the form:

$$ g_2(\tau) = 1 + A \exp\left(-\left(\frac{\tau}{\tau_c}\right)^\beta\right) $$

However, in the notebook's `stretched_exponential` function (used for fitting $F(\tau)$), the form is adapted to:

$$ F(\tau) = A \exp\left(-\left(\Gamma \tau\right)^\beta\right) + B $$

Alternatively, it can be written as:

$$ F(\tau) = A \exp\left(-\left(\frac{\tau}{\tau_c}\right)^\beta\right) + B $$

Where:
- **$A$ (Amplitude):** This parameter represents the maximum correlation value, related to the speckle contrast $\beta$ (from $g_2(\tau) = 1 + \beta |F(\tau)|^2$) and the coherent fraction of the scattering. Ideally, for $F(\tau)$, it should approach 1 for a fully coherent signal.
- **$\Gamma$ (Relaxation Rate) or $\tau_c$ (Characteristic Relaxation Time):**
    - $\Gamma$ is the characteristic decay rate, measured in $s^{-1}$.
    - $\tau_c = 1/\Gamma$ is the characteristic relaxation time, measured in seconds. It represents the timescale over which the correlation decays by a certain factor.
- **$\beta$ (Stretching Exponent):** This dimensionless parameter describes the shape of the decay:
    - If $\beta = 1$, the decay is a simple exponential, characteristic of a single, well-defined relaxation process (e.g., Brownian motion of monodisperse particles).
    - If $\beta < 1$, the decay is "stretched," indicating a distribution of relaxation times, common in systems with heterogeneous dynamics or glassy behavior.
    - If $\beta > 1$, the decay is "compressed," which can sometimes occur but is less common and might indicate cooperative dynamics or a failure of the model for that specific system.
- **$B$ (Baseline/Offset):** This parameter accounts for any constant offset in the correlation function at long lag times. Ideally, for $F(\tau)$, it should approach 0, indicating full decorrelation.

**Fitting Process (`fit_F_corr` function):**
- The `lmfit` library is used for non-linear curve fitting. It provides robust tools for defining models, setting parameter bounds, and performing the fit.
- **Parameter Initialization:** Initial guesses for $A$, $\Gamma$, $B$, and $\beta$ are provided. Crucially, the function dynamically sets reasonable bounds for $A$ and $B$ based on the maximum and minimum values of the `F_data` to be fitted, which helps guide the optimization algorithm.
- **Optimization:** `lmfit.Model.fit` performs the optimization, attempting to find the parameters that best fit the data according to a least-squares criterion.
- **Results:** The function returns the fitted parameters (`A`, `Gamma`, `B`, `beta`), the characteristic time `tau_char`, and the R-squared value, which indicates the goodness of fit. It also reports if any parameters hit their defined boundaries, which can suggest issues with the fit or parameter ranges.

In [ ]:
# Calculate time lags (same for all ROIs)
frame_time_s = (exposure_time + readout_time) / 1000.0  # Convert ms to seconds
frame_time_s

### Two-time correlation functions

In [ ]:
# TWO-TIME CORRELATION FUNCTIONS

print("TWO-TIME CORRELATION ANALYSIS")
print("=" * 60 + "\n")

# Analysis parameters

frames_for_analysis = 2700  # Adjust this as needed 2700
rois_for_analysis = [smd_0, smd_1, smd_2, smd_3]  # Add/remove ROIs as needed
roi_names = ['ROI_0', 'ROI_1', 'ROI_2', 'ROI_3']

frames_for_analysis = min(frames_for_analysis, smd_0.shape[0])
analysis_time = frames_for_analysis * frame_time_s

# Ensure num_bufs is even for two_time_corr
num_bufs_two_time = frames_for_analysis if frames_for_analysis % 2 == 0 else frames_for_analysis - 1

print(f"Analysis parameters\n" + "-" * 30)
print(f"Frames to analyze: {frames_for_analysis}")
print(f"ROIs to analyze: {roi_names}")
print(f"Frame time: {frame_time_s:.4f} s")
print(f"Analysis time range: 0 to {analysis_time:.1f} s")

# Two-time g2 alculation

print(f"\n🕐 Starting two-time g2 calculation...")

# Storage for results
g2_two_time_results = []
time_lags_results = []
computation_times = []

for i, (roi_data, roi_name) in enumerate(zip(rois_for_analysis, roi_names)):
    print(f"\n--- {roi_name} ---")

    # Prepare data
    analysis_data = roi_data[:frames_for_analysis]
    height, width = analysis_data.shape[1], analysis_data.shape[2]

    print(f"Data shape: {analysis_data.shape}")
    print(f"Pixels: {height * width}")

    # Create labeled array
    labeled_array = np.ones((height, width), dtype=int)

    # Calculate two-time correlation
    print(f"Calculating...")
    start_time = datetime.now()

    g2_two_time, lag_steps, _state = corr.two_time_corr(
        labeled_array,
        analysis_data,
        frames_for_analysis,
        num_bufs_two_time, # Changed from frames_for_analysis to ensure even number
        num_levels=1
    )

    computation_time = (datetime.now() - start_time).total_seconds()

    # Calculate time lags
    time_lags = lag_steps * frame_time_s

    print(f"Complete! Time: {computation_time:.1f}s")
    print(f"Result shape: {g2_two_time.shape}")
    print(f"g2 range: {np.min(g2_two_time[0]):.6f} to {np.max(g2_two_time[0]):.6f}")

    # Store results
    g2_two_time_results.append(g2_two_time)
    time_lags_results.append(time_lags)

print(f"\n✅ Two-time g2 calculation complete!")
print(f"Results stored as: g2_two_time_results")

In [ ]:
# Plotting two-time g2's (new)

print("Plotting two-time correlation functions ...\n")

# Plotting parameters
vmin_percentile = 0.1
vmax_percentile = 90

# Create figure with three subplots
fig, axes = plt.subplots(1, 4, figsize=(23, 5))

for i, (g2_two_time, time_lags, roi_name) in enumerate(zip(g2_two_time_results, time_lags_results, roi_names)):

    g2_matrix = g2_two_time[0]

    # Calculate contrast values once per ROI
    vmin = np.percentile(g2_matrix, vmin_percentile)
    vmax = np.percentile(g2_matrix, vmax_percentile)

    im = axes[i].imshow(g2_matrix, origin='lower', aspect='auto', cmap='viridis',
                       vmin=vmin, vmax=vmax)
    axes[i].set_xlabel('Time Index $t_2$')
    axes[i].set_ylabel('Time Index $t_1$')
    axes[i].set_title(f'{roi_name}: Two-Time $g_2(t_1,t_2)$')

    # Add colorbar to each subplot
    plt.colorbar(im, ax=axes[i])

plt.tight_layout()
# filename = f'{np.mean(temperature):.0f}K_two_time_corr.png'
# plt.savefig(os.path.join(results_path, filename), dpi=300, bbox_inches='tight')
plt.show()

print(f"Plotting percentile values: {vmin_percentile}% to {vmax_percentile}%")

In [ ]:
# Aging analysis

print("Diagonal aging analysis")
print("-" * 60)

fig, axes = plt.subplots(1, len(roi_names), figsize=(7*len(roi_names), 6))
if len(roi_names) == 1:
    axes = [axes]

aging_summary = []

for i, (g2_two_time, time_lags, roi_name) in enumerate(zip(g2_two_time_results, time_lags_results, roi_names)):

    # Extract diagonal
    g2_matrix = g2_two_time[0]
    diagonal = np.diag(g2_matrix)
    time_points = np.arange(len(diagonal)) * frame_time_s

    # Windowed analysis for robust statistics
    window_size = 60
    n_windows = len(diagonal) // window_size

    if n_windows >= 2:
        # Calculate windowed means
        windowed_means = np.array([np.mean(diagonal[i*window_size:(i+1)*window_size])
                                  for i in range(n_windows)])
        window_centers = np.array([(i + 0.5) * window_size * frame_time_s
                                  for i in range(n_windows)])

        # Linear fit for aging rate
        slope, intercept, r_value, p_value, std_err = stats.linregress(window_centers, windowed_means)
        relative_change = (window_centers[-1] - window_centers[0]) * slope / np.mean(windowed_means)

        aging_info = {
            'roi': roi_name,
            'aging_rate': slope,
            'relative_change': relative_change,
            'r_squared': r_value**2
        }
        aging_summary.append(aging_info)

        # Print summary
        print(f"{roi_name}: Rate: {slope:.2e}/s ({relative_change*100:+.2f}% change), R²: {r_value**2:.3f}")

        # Plot with windowed data for clarity
        axes[i].plot(time_points, diagonal, 'b-', alpha=0.3, linewidth=0.5, label='Raw diagonal')
        axes[i].plot(window_centers, windowed_means, 'ro-', markersize=4, linewidth=2, label='Windowed mean')

        # Add trend line
        trend_line = intercept + slope * window_centers
        axes[i].plot(window_centers, trend_line, 'r--', linewidth=2, label=f'Trend: {slope:.2e}/s')

    else:
        print(f"{roi_name}: Insufficient data for windowed analysis")
        axes[i].plot(time_points, diagonal, 'b-', alpha=0.7, label='Raw diagonal')

    axes[i].set_xlabel('Time (s)')
    axes[i].set_ylabel('g₂(t,t)')
    axes[i].set_title(f'{roi_name}: Aging Analysis')
    axes[i].grid(True, alpha=0.3)
    axes[i].legend()

plt.suptitle('Diagonal Aging: Identifying Stable Periods', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Off-diagonal analysis

print("Off-diagonal stationarity analysis")
print("-" * 60)

# Analysis parameters
offsets = [10, 33, 100, 300, 900]  # frames
colors = ['orange', 'green', 'red', 'purple', 'brown']
window_size = 60

fig, axes = plt.subplots(1, len(roi_names), figsize=(7*len(roi_names), 6))
if len(roi_names) == 1:
    axes = [axes]

stationarity_summary = []

for i, (g2_two_time, time_lags, roi_name) in enumerate(zip(g2_two_time_results, time_lags_results, roi_names)):

    g2_matrix = g2_two_time[0]
    diagonal = np.diag(g2_matrix)
    n_windows = len(diagonal) // window_size

    if n_windows >= 2:
        # Window setup
        window_centers = np.array([(j + 0.5) * window_size * frame_time_s
                                  for j in range(n_windows)])
        diagonal_means = np.array([np.mean(diagonal[j*window_size:(j+1)*window_size])
                                  for j in range(n_windows)])

        # Plot diagonal (reference)
        # axes[i].plot(window_centers, diagonal_means, 'bo-', linewidth=2.5,
        #             markersize=5, alpha=0.8, label='Diagonal g₂(t,t)')

        # Analyze off-diagonals
        offset_results = {}

        for j, offset in enumerate(offsets):
            if offset < min(g2_matrix.shape) - offset:
                off_diag = np.diag(g2_matrix, k=offset)

                if len(off_diag) >= window_size:
                    off_diag_windows = len(off_diag) // window_size
                    max_windows = min(off_diag_windows, n_windows)

                    off_diag_means = np.array([np.mean(off_diag[k*window_size:(k+1)*window_size])
                                              for k in range(max_windows)])
                    off_window_centers = window_centers[:len(off_diag_means)]

                    if len(off_diag_means) >= 3:
                        # Calculate STABILITY metrics
                        slope, intercept, r_value, p_value, std_err = stats.linregress(off_window_centers, off_diag_means)

                        relative_change = (off_window_centers[-1] - off_window_centers[0]) * slope / np.mean(off_diag_means) * 100
                        cv = np.std(off_diag_means) / np.mean(off_diag_means) * 100   # Coefficient of variation, in percentile
                        trend = slope * off_window_centers + intercept
                        detrended_cv = np.std(off_diag_means - trend) / np.mean(off_diag_means) * 100

                        offset_results[offset] = {
                            'relative_change': relative_change,
                            'cv': cv,
                            'detrended_cv': detrended_cv,
                            'r_squared': r_value**2
                        }

                        # Plot off-diagonal
                        axes[i].plot(off_window_centers, off_diag_means,
                                   color=colors[j % len(colors)], marker='s', linewidth=2,
                                   markersize=4, alpha=0.8,
                                   label=f'Offset {offset} ({offset * frame_time_s:.0f}s)')

        # Store results for this ROI
        if offset_results:
            stationarity_summary.append({
                'roi': roi_name,
                'offset_results': offset_results
            })

    else:
        print(f"{roi_name}: Insufficient data for analysis")

    axes[i].set_xlabel('Time (s)')
    axes[i].set_ylabel('Windowed g₂ Mean')
    axes[i].set_title(f'{roi_name}: Stationarity Check')
    axes[i].legend(fontsize=9)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Off-Diagonal Analysis: Validating Stationarity Assumptions', fontsize=14)
plt.tight_layout()
plt.show()

# Display results in tables

# Table 1: Relative Change (systematic trend) - using slope method
print(f"\nRelative Change (systematic trend):")
header = f"{'ROI':<6} Offset =  "
for offset in offsets:
    header += f"{offset:2d} ({offset * frame_time_s:2.0f}s)"
    if offset != offsets[-1]:
        header += "   "
print(header)
print("-" * len(header))

for result in stationarity_summary:
    roi_name = result['roi']
    offset_results = result['offset_results']

    row = f"{roi_name:<14}"
    for offset in offsets:
        if offset in offset_results:
            relative_change = offset_results[offset]['relative_change']
            row += f"{relative_change:+8.2f}%   "
        else:
            row += f"{'N/A':>8}%   "
    print(row)

# Table 2: Coefficient of Variation (CV) - overall stability
print(f"\nCoefficient of Variation (overall stability):")
header = f"{'ROI':<6} Offset =  "
for offset in offsets:
    header += f"{offset:2d} ({offset * frame_time_s:2.0f}s)"
    if offset != offsets[-1]:
        header += "   "
print(header)
print("-" * len(header))

for result in stationarity_summary:
    roi_name = result['roi']
    offset_results = result['offset_results']

    row = f"{roi_name:<14}"
    for offset in offsets:
        if offset in offset_results:
            cv = offset_results[offset]['cv']
            row += f"{cv:8.2f}%   "
        else:
            row += f"{'N/A':>8}%   "
    print(row)

# Table 3: Detrended CV - pure stationarity (stability after removing trends)
print(f"\nDetrended CV (pure stationarity):")
header = f"{'ROI':<6} Offset =  "
for offset in offsets:
    header += f"{offset:2d} ({offset * frame_time_s:2.0f}s)"
    if offset != offsets[-1]:
        header += "   "
print(header)
print("-" * len(header))

for result in stationarity_summary:
    roi_name = result['roi']
    offset_results = result['offset_results']

    row = f"{roi_name:<14}"
    for offset in offsets:
        if offset in offset_results:
            detrended_cv = offset_results[offset]['detrended_cv']
            row += f"{detrended_cv:8.2f}%   "
        else:
            row += f"{'N/A':>8}%   "
    print(row)

### One-time correlation functions

In [ ]:
# Calculate g2 correlation functions for all ROIs - WHOLE WINDOW
print("CALCULATING ONE-TIME CORRELATION (FFT METHOD)")
print("=" * 60)

# Calculate g2 and F(τ) for each ROI using FFT method
rois_data = [smd_0, smd_1, smd_2, smd_3]
roi_names = ['ROI 0', 'ROI 1', 'ROI 2', 'ROI 3']
start_frame = [0, 0, 0, 0]
end_frame   = [-1, -1, -1, -1]
correlation_results = []

for roi_data, roi_name in zip(rois_data, roi_names):
    print(f"\n--- {roi_name} ---")
    results = calculate_corr_fft(roi_data, identifier=roi_name)
    correlation_results.append(results)

# Store individual arrays for easy access
g2_0, g2_1, g2_2, g2_3 = [results['g2'] for results in correlation_results]
F_0, F_1, F_2, F_3 = [results['F'] for results in correlation_results]

# Get time lags from the function results (all should be the same)
time_lags = correlation_results[0]['lag_time_s']
print(f"\nTime lag parameters:")
print(f"    Frame interval: {time_lags[1] - time_lags[0]:.4f} s")
print(f"    Maximum lag time: {time_lags[-1]:.2f} s")

# Plot g2 and F functions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# g2 comparison
ax1.semilogx(time_lags[1:], g2_0[1:], 'o-', markersize=2, linewidth=1, label='ROI 0', alpha=0.8)
ax1.semilogx(time_lags[1:], g2_1[1:], 's-', markersize=2, linewidth=1, label='ROI 1', alpha=0.8)
ax1.semilogx(time_lags[1:], g2_2[1:], '^-', markersize=2, linewidth=1, label='ROI 2', alpha=0.8)
ax1.semilogx(time_lags[1:], g2_3[1:], 'd-', markersize=2, linewidth=1, label='ROI 3', alpha=0.8)
ax1.set_xlabel('Time Lag (s)', fontsize=14)
ax1.set_ylabel('g2(tau)', fontsize=14)
ax1.set_title('One-time Correlation Functions (FFT)', fontsize=16)
ax1.grid(True, alpha=0.5)
ax1.legend()

# F comparison
ax2.semilogx(time_lags[2:], F_0[1:], 'o-', markersize=2, linewidth=1, label='ROI 0', alpha=0.8)
ax2.semilogx(time_lags[2:], F_1[1:], 's-', markersize=2, linewidth=1, label='ROI 1', alpha=0.8)
ax2.semilogx(time_lags[2:], F_2[1:], '^-', markersize=2, linewidth=1, label='ROI 2', alpha=0.8)
ax2.semilogx(time_lags[2:], F_3[1:], 'd-', markersize=2, linewidth=1, label='ROI 3', alpha=0.8)
ax2.set_xlabel('Time Lag (s)', fontsize=14)
ax2.set_ylabel('F(tau)', fontsize=14)
ax2.set_title('Intermediate Scattering Functions (FFT)', fontsize=16)
ax2.grid(True, alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\n✅ All g2 and F calculations complete using FFT method!")
print(f"Results stored as: correlation_results")

In [ ]:
# Plot F(tau) functions with cutoff

cutoff_frame = 2000   # Adjust as needed.
cutoff_idx = min(cutoff_frame - 1, len(F_0))  # F is one point shorter than g2
actual_cutoff_time_s = time_lags[cutoff_idx + 1]

print(f"F(tau) plot with cutoff at {actual_cutoff_time_s:.1f} s (frame {cutoff_frame})")

plt.figure(figsize=(8, 6))

plt.semilogx(time_lags[2:cutoff_idx+2], F_0[1:cutoff_idx+1], 'o-', markersize=1.5, linewidth=1,
             label='ROI 0', alpha=0.5)
plt.semilogx(time_lags[2:cutoff_idx+2], F_1[1:cutoff_idx+1], 's-', markersize=1.5, linewidth=1,
             label='ROI 1', alpha=0.5)
plt.semilogx(time_lags[2:cutoff_idx+2], F_2[1:cutoff_idx+1], '^-', markersize=1.5, linewidth=1,
             label='ROI 2', alpha=0.5)
plt.semilogx(time_lags[2:cutoff_idx+2], F_3[1:cutoff_idx+1], 'd-', markersize=1.5, linewidth=1,
             label='ROI 3', alpha=0.5)

plt.xlabel('Time Lag (s)', fontsize=14)
plt.ylabel('F(tau)', fontsize=14)
plt.title(f'Intermediate Scattering Functions (FFT, cutoff at {actual_cutoff_time_s:.1f} s)', fontsize=16)
plt.grid(True, alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

## Analyze Dynamics from One-Time Correlation Functions


### FFT Analysis of g2 - Quality Control (Skipped for now)

Fourier transform the g2 data to identify any unusual features or periodic artifacts

### One-time correlation via multi-tau and fitting

In [ ]:
print("CALCULATING MULTITAU CORRELATION FOR FITTING")
print("=" * 60 + '\n')

# Calculate multitau correlation for ROI 0-3 (extended test phase)
# rois_test = [smd_0[360:], smd_1[360:], smd_2[360:], smd_3[360:]]
rois_test = [smd_0, smd_1, smd_2, smd_3]
roi_names_test = ['ROI 0', 'ROI 1', 'ROI 2', 'ROI 3']
multitau_results = []

# Calculate multitau correlation for test ROIs
for roi_data, roi_name in zip(rois_test, roi_names_test):
    print(f"Processing {roi_name}...")

    # Calculate using multitau method with optimized parameters for fitting
    results = calculate_corr_multitau(
        data=roi_data,
        identifier=roi_name,
        num_levels=9,      # Extended time range
        num_bufs=10        # Better statistics
    )

    multitau_results.append(results)

# Extract data for plotting
g2_0_mt, g2_1_mt, g2_2_mt, g2_3_mt = [results['g2'] for results in multitau_results]
F_0_mt, F_1_mt, F_2_mt, F_3_mt = [results['F'] for results in multitau_results]
time_lags_mt = multitau_results[0]['lag_time_s']  # Same for all ROIs

# Plot decay curves to visualize results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# g2 decay curves
ax1.semilogx(time_lags_mt, g2_0_mt, 'o-', markersize=2, linewidth=1.5, label='ROI 0', alpha=0.8)
ax1.semilogx(time_lags_mt, g2_1_mt, 's-', markersize=2, linewidth=1.5, label='ROI 1', alpha=0.8)
ax1.semilogx(time_lags_mt, g2_2_mt, '^-', markersize=2, linewidth=1.5, label='ROI 2', alpha=0.8)
ax1.semilogx(time_lags_mt, g2_3_mt, 'd-', markersize=2, linewidth=1.5, label='ROI 3', alpha=0.8)
ax1.set_xlabel('Time Lag (s)', fontsize=12)
ax1.set_ylabel('g2(tau)', fontsize=12)
ax1.set_title('Multitau g2 Decay Curves', fontsize=14)
ax1.grid(True, alpha=0.5)
ax1.legend()

# F(τ) decay curves
ax2.semilogx(time_lags_mt, F_0_mt, 'o-', markersize=2, linewidth=1.5, label='ROI 0', alpha=0.8)
ax2.semilogx(time_lags_mt, F_1_mt, 's-', markersize=2, linewidth=1.5, label='ROI 1', alpha=0.8)
ax2.semilogx(time_lags_mt, F_2_mt, '^-', markersize=2, linewidth=1.5, label='ROI 2', alpha=0.8)
ax2.semilogx(time_lags_mt, F_3_mt, 'd-', markersize=2, linewidth=1.5, label='ROI 3', alpha=0.8)
ax2.set_xlabel('Time Lag (s)', fontsize=12)
ax2.set_ylabel('F(tau)', fontsize=12)
ax2.set_title('Multitau F Decay Curves', fontsize=14)
ax2.grid(True, alpha=0.5)
ax2.legend()

plt.tight_layout()
filename = f'{np.mean(temperature):.0f}K_one_time_corr.png'
plt.savefig(os.path.join(results_path, filename), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
i = 2

roi_fit = fit_F_corr(
    F_data=multitau_results[i]['F'],
    time_lags=multitau_results[i]['lag_time_s'],
    roi_name=f'ROI {i}',
    start_idx=3,
    end_idx=-3,
    initial_gamma=5e-4,
    plot=True
)

In [ ]:
# Batch fitting

all_fits = {}
start_idx = [3, 3, 3, 3]
end_idx   = [-2, -5, -3, -5]

for i, result in enumerate(multitau_results):
    roi_name = f'ROI {i}'

    fit_result = fit_F_corr(
        F_data=result['F'],
        time_lags=result['lags'],
        roi_name=roi_name,
        start_idx=start_idx[i],
        end_idx=end_idx[i],
        initial_gamma=5e-4,
        plot=False
    )

    all_fits[roi_name] = fit_result

# Comparison plot 1: F(tau) data and fits
plt.figure(figsize=(9, 6))

for i, (roi_name, fit_result) in enumerate(all_fits.items()):
    if fit_result['success']:
        color = f'C{i}'

        all_data = multitau_results[i]['F']
        all_times = multitau_results[i]['lags']
        plt.semilogx(all_times, all_data, 'o', color=color, ms=1.5, alpha=0.5,
                    label=f'{roi_name} all data')
        plt.semilogx(fit_result['fit_times'], fit_result['fit_data'], 'o',
                    color=color, ms=2.5, alpha=0.8, label=f'{roi_name} fitted')
        plt.semilogx(fit_result['fit_times'], fit_result['fit_curve'],
                    '-', color=color, linewidth=2.5, alpha=0.8)

plt.xlabel('tau (s)', fontsize=12)
plt.ylabel('F(tau)', fontsize=12)
plt.ylim(bottom=0.99)
plt.title('F(tau) Comparison - All ROIs', fontsize=14)
plt.grid(True, alpha=0.3, which='major')
plt.grid(True, alpha=0.15, which='minor')
plt.minorticks_on()
plt.legend()
plt.tight_layout()
filename = f'{np.mean(temperature):.0f}K_fitting_1.png'
plt.savefig(os.path.join(results_path, filename), dpi=300, bbox_inches='tight')
plt.show()

# Comparison plot 2: Parameters
successful_fits = [fit for fit in all_fits.values() if fit['success']]

if successful_fits:
    roi_numbers = [int(fit['roi_name'].split()[1]) for fit in successful_fits]
    tau_chars = [fit['tau_char'] for fit in successful_fits]
    betas = [fit['beta'] for fit in successful_fits]

    plt.figure(figsize=(8, 4))

    plt.subplot(1, 2, 1)
    plt.plot(roi_numbers, tau_chars, 'o-', markersize=6, linewidth=2)
    plt.xlabel('ROI Number', fontsize=12)
    plt.ylabel('tau_c (s)', fontsize=12)
    plt.title('Characteristic Time vs ROI', fontsize=12)
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(roi_numbers, betas, 'o-', markersize=6, linewidth=2)
    plt.xlabel('ROI Number', fontsize=12)
    plt.ylabel('beta', fontsize=12)
    plt.title('Stretching Exponent vs ROI', fontsize=12)
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    filename = f'{np.mean(temperature):.0f}K_fitting_2.png'
    plt.savefig(os.path.join(results_path, filename), dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# Process fit results for data manager storage

fit_results_processed = {}
for roi_name, fit_result in all_fits.items():
    # Get ROI index to access the corresponding start/end indices
    roi_idx = int(roi_name.split()[1])  # Extract number from 'ROI 0', 'ROI 1', etc.

    if fit_result['success']:
        fit_results_processed[roi_name] = {
            'A': fit_result['A'],
            'Gamma': fit_result['Gamma'],
            'B': fit_result['B'],
            'beta': fit_result['beta'],
            'tau_char': fit_result['tau_char'],
            'r_squared': fit_result['r_squared'],
            'method': 'multitau',
            'success': True,
            'start_idx': start_idx[roi_idx],
            'end_idx': end_idx[roi_idx],
            'n_points_fitted': len(fit_result['fit_data'])
        }
        print(f"{roi_name}: τ_c = {fit_result['tau_char']:.1f}s, β = {fit_result['beta']:.3f}, R² = {fit_result['r_squared']:.4f} "
              f"(idx {start_idx[roi_idx]}:{end_idx[roi_idx]}, {len(fit_result['fit_data'])} pts)")
    else:
        fit_results_processed[roi_name] = {
            'success': False,
            'error': fit_result.get('error', 'Unknown error'),
            'method': 'multitau',
            # Store attempted fitting range even for failed fits
            'start_idx': start_idx[roi_idx],
            'end_idx': end_idx[roi_idx]
        }
        print(f"{roi_name}: Fit failed (attempted idx {start_idx[roi_idx]}:{end_idx[roi_idx]})")

print(f"\n✅ Fit results processed and results stored as: fit_results_processed")


### Save to data manager


In [ ]:
print("SAVE RESULTS TO DATA MANAGER")
print("=" * 60)

# Extract current temperature and metadata
current_temperature = np.mean(temperature)
print(f"Current temperature: {current_temperature:.1f} K")

# Prepare metadata dictionary
metadata = {
    'file_index': file_index,
    'total_frames': len(data),
    'exposure_time': exposure_time,
    'readout_time': readout_time,
    'beam_energy': np.mean(beamline_energy),
    'sample_detector_distance': sample_detector_distance
}

# Prepare ROI definitions
roi_definitions = {
    'ROI 0': {'smd_shape': smd_0.shape, 'coordinates': (roi_0_x, roi_0_y, roi_0_width, roi_0_height)},
    'ROI 1': {'smd_shape': smd_1.shape, 'coordinates': (roi_1_x, roi_1_y, roi_1_width, roi_1_height)},
    'ROI 2': {'smd_shape': smd_2.shape, 'coordinates': (roi_2_x, roi_2_y, roi_2_width, roi_2_height)},
    'ROI 3': {'smd_shape': smd_3.shape, 'coordinates': (roi_3_x, roi_3_y, roi_3_width, roi_3_height)}
}

# Prepare two-time correlation results
two_time_g2_results = {
    'ROI 0': {'g2_matrix': g2_two_time_results[0][0], 'time_lags': time_lags_results[0]},
    'ROI 1': {'g2_matrix': g2_two_time_results[1][0], 'time_lags': time_lags_results[1]},
    'ROI 2': {'g2_matrix': g2_two_time_results[2][0], 'time_lags': time_lags_results[2]},
    'ROI 3': {'g2_matrix': g2_two_time_results[3][0], 'time_lags': time_lags_results[3]}
}

# Prepare one-time correlation results (FFT method)
one_time_corr_fft = {
    'ROI 0': [correlation_results[0]],  # List format for consistency
    'ROI 1': [correlation_results[1]],
    'ROI 2': [correlation_results[2]],
    'ROI 3': [correlation_results[3]],
}

# Prepare one-time correlation results (Multitau method)
one_time_corr_multitau = {
    'ROI 0': [multitau_results[0]],
    'ROI 1': [multitau_results[1]],
    'ROI 2': [multitau_results[2]],
    'ROI 3': [multitau_results[3]]
}

# Prepare fit results
fit_results=fit_results_processed

print("All data including fit results are prepared.")

In [ ]:
# Load existing data manager from Drive

print("Loading Data Manager from Drive ...")

try:
    dm = load_xpcs_manager_from_drive('/content/drive/Shareddrives/X Lab UCSD/Beamtimes/2025-04-16_ALS_COSMIC_FNS w Shan/Data/Francis_XPCS/Results/xpcs_data_manager_0.pkl')
    print("✅ Loaded existing data manager from Drive")
    dm.get_summary()

except FileNotFoundError:
    print("⚠️  No existing data manager found. Using current data manager.")
    dm.get_summary()

except Exception as e:
    print(f"❌ Error loading data manager: {e}")
    print("   Using current data manager instead.")
    dm.get_summary()

print("\nReady to add new temperature data!")

In [ ]:
# Store everything in data manager (including fit results!)
dm.store_temperature_data(
    temperature=current_temperature,
    metadata=metadata,
    roi_definitions=roi_definitions,
    two_time_g2_results=two_time_g2_results,
    one_time_corr_fft=one_time_corr_fft,
    one_time_corr_multitau=one_time_corr_multitau,
    fit_results=fit_results_processed
)

# Check what's stored
dm.get_summary()


In [ ]:
# Save to Google Drive
dm.save_to_drive(drive_path=os.path.join(results_path), filename= 'xpcs_data_manager_0.pkl')


# MULTI-TEMPERATURE ANALYSIS

In [ ]:
# Load existing data manager from Drive

print("Loading Data Manager from Drive ...")

try:
    dm = load_xpcs_manager_from_drive('/content/drive/Shareddrives/X Lab UCSD/Beamtimes/2025-04-16_ALS_COSMIC_FNS w Shan/Data/Francis_XPCS/Results/xpcs_data_manager_0.pkl')
    print("✅ Loaded existing data manager from Drive")
    dm.get_summary()

except FileNotFoundError:
    print("⚠️  No existing data manager found. Using current data manager.")
    dm.get_summary()

except Exception as e:
    print(f"❌ Error loading data manager: {e}")
    print("   Using current data manager instead.")
    dm.get_summary()

print("\nXPCS results stored in dm.")

In [ ]:
print("COMPARE CORRELATION IN DIFFERENT TEMPERATURES.")
print("=" * 60)

print("Extracting data from Data Manager ...\n")

# Get all available temperatures
available_temperatures = dm.get_all_temperatures()
print(f"Available temperatures: {available_temperatures}")

# Initialize data structure
all_temp_data = {}

# Extract multitau correlation data for each temperature and ROI
for temp in available_temperatures:
    print(f"\n🌡️  Processing {temp}...")
    temp_data = {}

    # Get data for this temperature
    temp_dataset = dm.data[f'{temp}K']

    # Extract multitau data for each ROI
    for roi_name in temp_dataset.keys():
        if roi_name != 'metadata':  # Skip metadata entry
            roi_data = temp_dataset[roi_name]

            # Get multitau correlation data
            if 'one_time_corr_multitau' in roi_data:
                multitau_data = roi_data['one_time_corr_multitau'][0]  # First (and only) multitau result

                temp_data[roi_name] = {
                    'F': multitau_data['F'],
                    'lags': multitau_data['lags'],
                }

                print(f"   {roi_name}: F-data shape = {multitau_data['F'].shape}, "
                      f"lags shape = {multitau_data['lags'].shape}")
            else:
                print(f"   ⚠️  {roi_name}: No multitau data found")

    all_temp_data[temp] = temp_data

print(f"\n✅ Data extraction complete!")
print(f"   Structure: {len(available_temperatures)} temperatures, "
      f"{len(list(all_temp_data.values())[0])} ROIs per temperature")


In [ ]:
print("Extracting fit results from Data Manager ...")

# Initialize fitting parameter structures
fitting_params = {}  # {temp: {roi: {'start_idx': x, 'end_idx': y}}}

# Extract fitting parameters from stored fit results
for temp in available_temperatures:
    print(f"\n🌡️  {temp}K:")
    temp_params = {}

    # Get stored fit results for this temperature
    temp_dataset = dm.data[f'{temp}K']

    for roi_name in temp_dataset.keys():
        if roi_name != 'metadata':
            roi_data = temp_dataset[roi_name]

            # Check if fit results exist
            if 'fit_results' in roi_data:
                fit_result = roi_data['fit_results']

                if fit_result['success']:
                    temp_params[roi_name] = {
                        'start_idx': fit_result['start_idx'],
                        'end_idx': fit_result['end_idx'],
                        'initial_gamma': 5e-4
                    }
                    print(f"   {roi_name}: indices [{fit_result['start_idx']}:{fit_result['end_idx']}]")
                else:
                    temp_params[roi_name] = {
                        'start_idx': 3,
                        'end_idx': -5,
                        'initial_gamma': 5e-4
                    }
                    print(f"   {roi_name}: Using default indices [3:-5] (previous fit failed)")
            else:
                temp_params[roi_name] = {
                    'start_idx': 3,
                    'end_idx': -5,
                    'initial_gamma': 5e-4
                }
                print(f"   {roi_name}: Using default indices [3:-5] (no stored results)")

    fitting_params[temp] = temp_params

print(f"\n✅ Fitting parameters extracted!")

In [ ]:
print("BATCH FITTING ACROSS ALL TEMPERATURES")
print("=" * 60)

# Perform batch fitting across all temperatures and ROIs
all_temp_fits = {}  # {temp: {roi: fit_result}}

for temp in available_temperatures:
    print(f"\n🌡️  Fitting {temp}K...")
    temp_fits = {}

    for roi_name in all_temp_data[temp].keys():
        # Get data and fitting parameters
        F_data = all_temp_data[temp][roi_name]['F'].flatten()  # Flatten (49,1) to (49,)
        time_lags = all_temp_data[temp][roi_name]['lags']
        params = fitting_params[temp][roi_name]

        # Perform fitting with stored parameters
        fit_result = fit_F_corr(
            F_data=F_data,
            time_lags=time_lags,
            roi_name=roi_name,
            start_idx=params['start_idx'],
            end_idx=params['end_idx'],
            initial_gamma=params['initial_gamma'],
            plot=False
        )

        temp_fits[roi_name] = fit_result

        if fit_result['success']:
            print(f"   {roi_name}: τ_c={fit_result['tau_char']:.2f}s, β={fit_result['beta']:.3f}")
        else:
            print(f"   {roi_name}: Fit failed")

    all_temp_fits[temp] = temp_fits

print(f"\n✅ All fitting complete!")
print(f"📊 Results stored in: all_temp_fits")

In [ ]:
print("CREATING F(τ) COMPARISON PLOTS")
print("=" * 60)

# Get ROI names
roi_names = list(all_temp_data[list(available_temperatures)[0]].keys())
n_rois = len(roi_names)

# Create subplots (one per ROI)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

colors = ['C0', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8']

for roi_idx, roi_name in enumerate(roi_names):
    ax = axes[roi_idx]

    for temp_idx, temp in enumerate(available_temperatures):
        color = colors[temp_idx]

        # Get data
        F_data = all_temp_data[temp][roi_name]['F'].flatten()
        time_lags = all_temp_data[temp][roi_name]['lags']

        # Plot all data points
        ax.semilogx(time_lags, F_data, 'o', color=color, ms=1.5, alpha=0.5,
                   label=f'{temp}K data')

        # Plot fitted curve if successful
        fit_result = all_temp_fits[temp][roi_name]
        if fit_result['success']:
            ax.semilogx(fit_result['fit_times'], fit_result['fit_data'], 'o',
                       color=color, ms=2.5, alpha=0.8)
            ax.semilogx(fit_result['fit_times'], fit_result['fit_curve'],
                       '-', color=color, linewidth=2.5, alpha=0.8,
                       label=f'{temp}K fit')

    ax.set_xlabel('τ (s)', fontsize=10)
    ax.set_ylabel('F(τ)', fontsize=10)
    ax.set_ylim(bottom=0.97, top=1.005)
    ax.set_title(f'{roi_name} - Temperature Comparison', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
filename = 'multi_temperature_F_tau_comparison.png'
plt.savefig(os.path.join(results_path, filename), dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
print("CREATING PARAMETERS vs TEMPERATURE PLOTS")
print("=" * 60)

# Extract parameters for successful fits
roi_names = list(all_temp_data[list(available_temperatures)[0]].keys())
beta_data = {roi: [] for roi in roi_names}
tau_char_data = {roi: [] for roi in roi_names}

# Collect data for each temperature and ROI
for temp in available_temperatures:
    for roi_name in roi_names:
        fit_result = all_temp_fits[temp][roi_name]

        if fit_result['success']:
            beta_data[roi_name].append(fit_result['beta'])
            tau_char_data[roi_name].append(fit_result['tau_char'])
        else:
            # Handle failed fits
            beta_data[roi_name].append(None)
            tau_char_data[roi_name].append(None)

# Remove None values and corresponding temperatures for plotting
for roi_name in roi_names:
    valid_indices = [i for i, val in enumerate(beta_data[roi_name]) if val is not None]
    beta_data[roi_name] = [beta_data[roi_name][i] for i in valid_indices]
    tau_char_data[roi_name] = [tau_char_data[roi_name][i] for i in valid_indices]

# Create comparison plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['C0', 'C1', 'C2', 'C3']
markers = ['o', 's', '^', 'D']

# Plot 1: Beta vs Temperature
ax1 = axes[0]
for roi_idx, roi_name in enumerate(roi_names):
    if beta_data[roi_name]:  # Only plot if we have data
        ax1.plot(available_temperatures, beta_data[roi_name],
                color=colors[roi_idx], marker=markers[roi_idx],
                markersize=8, linewidth=2, label=roi_name)

ax1.set_xlabel('Temperature (K)', fontsize=12)
ax1.set_ylabel('Beta (stretching exponent)', fontsize=12)
ax1.set_title('Stretching Exponent vs Temperature', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.legend()

# Plot 2: Tau_char vs Temperature
ax2 = axes[1]
for roi_idx, roi_name in enumerate(roi_names):
    if tau_char_data[roi_name]:  # Only plot if we have data
        ax2.plot(available_temperatures, tau_char_data[roi_name],
                color=colors[roi_idx], marker=markers[roi_idx],
                markersize=8, linewidth=2, label=roi_name)

ax2.set_xlabel('Temperature (K)', fontsize=12)
ax2.set_ylabel('τ_c (s)', fontsize=12)
ax2.set_title('Characteristic Time vs Temperature', fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
filename = 'parameters_vs_temperature.png'
plt.savefig(os.path.join(results_path, filename), dpi=300, bbox_inches='tight')
plt.show()

print(f"📁 Plot saved as: {filename}")

# Print numerical summary
print(f"\n📋 PARAMETER SUMMARY:")
print("-" * 50)
for temp in available_temperatures:
    print(f"\n{temp}K:")
    for roi_name in roi_names:
        fit_result = all_temp_fits[temp][roi_name]
        if fit_result['success']:
            print(f"  {roi_name}: β={fit_result['beta']:.3f}, "
                  f"τ_c={fit_result['tau_char']:.2f} s")
        else:
            print(f"  {roi_name}: Fit failed")

## Comparison between Two Sets of XPCS



In [ ]:
# Load existing data manager from Drive

print("Loading Data Manager from Drive ...")

# Initialize variables
dm0, dm1 = None, None
loading_successful = True

# Load Dataset 0 (older data)
print("\n📁 Loading Dataset 0 (older data)...")
try:
    dm0 = load_xpcs_manager_from_drive('/content/drive/Shareddrives/X Lab UCSD/Beamtimes/2025-04-16_ALS_COSMIC_FNS w Shan/Data/Francis_XPCS/Results/xpcs_data_manager_0.pkl')
    print("✅ Dataset 0 loaded successfully")
    dm0.get_summary()

except FileNotFoundError:
    print("❌ Dataset 0 file not found!")
    loading_successful = False

except Exception as e:
    print(f"❌ Error loading Dataset 0: {e}")
    loading_successful = False

# Load Dataset 1 (newer data)
print("\n📁 Loading Dataset 1 (newer data)...")
try:
    dm1 = load_xpcs_manager_from_drive('/content/drive/Shareddrives/X Lab UCSD/Beamtimes/2025-04-16_ALS_COSMIC_FNS w Shan/Data/Francis_XPCS/Results/xpcs_data_manager.pkl')
    print("✅ Dataset 1 loaded successfully")
    dm1.get_summary()

except FileNotFoundError:
    print("❌ Dataset 1 file not found!")
    loading_successful = False

except Exception as e:
    print(f"❌ Error loading Dataset 1: {e}")
    loading_successful = False

# Validation check
if loading_successful and dm0 is not None and dm1 is not None:
    print("\n✅ Both datasets loaded successfully!")
    print(f"Dataset 0 temperatures: {dm0.get_all_temperatures()}")
    print(f"Dataset 1 temperatures: {dm1.get_all_temperatures()}")
else:
    print("\n❌ Failed to load one or both datasets. Cannot proceed with comparison.")
    print("Please check file paths and ensure both .pkl files exist.")

In [ ]:
# Extract one_time_corr_multitau data from both datasets for comparison

print("Extracting multitau correlation data from both datasets...")

# Validation check before proceeding
if not loading_successful or dm0 is None or dm1 is None:
    print("❌ Cannot proceed - one or both datasets failed to load!")
    print("Please run the previous cell successfully first.")
else:
    print("✅ Both datasets available for extraction")

# Get available temperatures from both datasets
temps_dm0 = dm0.get_all_temperatures()
temps_dm1 = dm1.get_all_temperatures()

print(f"\n📊 Dataset Overview:")
print(f"Dataset 0 temperatures: {temps_dm0}")
print(f"Dataset 1 temperatures: {temps_dm1}")
print(f"Total unique temperatures: {sorted(set(temps_dm0 + temps_dm1))}")

# Initialize combined data structure
all_temp_corr_combined = {
    'dataset_0': {},
    'dataset_1': {},
    'metadata': {
        'dataset_0_temps': temps_dm0,
        'dataset_1_temps': temps_dm1,
        'total_datasets': len(temps_dm0) + len(temps_dm1)
    }
}

print(f"\n🔄 Extracting Dataset 0 (older data)...")
# Extract from Dataset 0
for temp in temps_dm0:
    print(f"  🌡️  Processing {temp}K...")
    temp_data = {}

    # Get data for this temperature from dm0
    temp_dataset = dm0.data[f'{temp}K']

    # Extract multitau data for each ROI
    roi_count = 0
    for roi_name in temp_dataset.keys():
        if roi_name != 'metadata':  # Skip metadata entry
            roi_data = temp_dataset[roi_name]

            # Get multitau correlation data
            if 'one_time_corr_multitau' in roi_data:
                multitau_data = roi_data['one_time_corr_multitau'][0]  # First (and only) multitau result

                temp_data[roi_name] = {
                    'F': multitau_data['F'],
                    'lags': multitau_data['lags'],
                    'dataset': 'dataset_0',
                    'temperature': temp
                }

                roi_count += 1
                print(f"     {roi_name}: F-data shape = {multitau_data['F'].shape}, "
                      f"lags shape = {multitau_data['lags'].shape}")
            else:
                print(f"     ⚠️  {roi_name}: No multitau data found")

    all_temp_corr_combined['dataset_0'][temp] = temp_data
    print(f"     ✅ Extracted {roi_count} ROIs for {temp}K")

print(f"\n🔄 Extracting Dataset 1 (newer data)...")
# Extract from Dataset 1
for temp in temps_dm1:
    print(f"  🌡️  Processing {temp}K...")
    temp_data = {}

    # Get data for this temperature from dm1
    temp_dataset = dm1.data[f'{temp}K']

    # Extract multitau data for each ROI
    roi_count = 0
    for roi_name in temp_dataset.keys():
        if roi_name != 'metadata':  # Skip metadata entry
            roi_data = temp_dataset[roi_name]

            # Get multitau correlation data
            if 'one_time_corr_multitau' in roi_data:
                multitau_data = roi_data['one_time_corr_multitau'][0]  # First (and only) multitau result

                temp_data[roi_name] = {
                    'F': multitau_data['F'],
                    'lags': multitau_data['lags'],
                    'dataset': 'dataset_1',
                    'temperature': temp
                }

                roi_count += 1
                print(f"     {roi_name}: F-data shape = {multitau_data['F'].shape}, "
                      f"lags shape = {multitau_data['lags'].shape}")
            else:
                print(f"     ⚠️  {roi_name}: No multitau data found")

    all_temp_corr_combined['dataset_1'][temp] = temp_data
    print(f"     ✅ Extracted {roi_count} ROIs for {temp}K")

# Summary of extraction
print(f"\n✅ Data extraction complete!")
print(f"📊 EXTRACTION SUMMARY:")
print(f"   Dataset 0: {len(all_temp_corr_combined['dataset_0'])} temperatures")
print(f"   Dataset 1: {len(all_temp_corr_combined['dataset_1'])} temperatures")

# Get ROI names for verification
if all_temp_corr_combined['dataset_0']:
    sample_temp_0 = list(all_temp_corr_combined['dataset_0'].keys())[0]
    rois_0 = list(all_temp_corr_combined['dataset_0'][sample_temp_0].keys())
    print(f"   Dataset 0 ROIs: {rois_0}")

if all_temp_corr_combined['dataset_1']:
    sample_temp_1 = list(all_temp_corr_combined['dataset_1'].keys())[0]
    rois_1 = list(all_temp_corr_combined['dataset_1'][sample_temp_1].keys())
    print(f"   Dataset 1 ROIs: {rois_1}")

# Total data points for fitting
total_data_points = (len(all_temp_corr_combined['dataset_0']) +
                    len(all_temp_corr_combined['dataset_1']))
print(f"   📈 Total data points for fitting: {total_data_points}")

print(f"\n🎯 Data structure created: 'all_temp_corr_combined'")
print(f"   Access pattern: all_temp_corr_combined['dataset_0'][temperature][roi_name]['F']")
print(f"   Ready for batch fitting analysis!")

In [ ]:
# Extract fitting parameters from both data managers

print("Extracting fitting parameters from both Data Managers ...")

# Check if we have the extracted correlation data
if 'all_temp_corr_combined' not in globals():
    print("❌ No extracted correlation data found!")
    print("Please run the data extraction cell first.")
else:
    print("✅ Found correlation data for parameter extraction")

# Initialize fitting parameter structures for both datasets
fitting_params_combined = {
    'dataset_0': {},  # {temp: {roi: {'start_idx': x, 'end_idx': y, 'initial_gamma': z}}}
    'dataset_1': {},  # {temp: {roi: {'start_idx': x, 'end_idx': y, 'initial_gamma': z}}}
    'metadata': {
        'default_params': {
            'start_idx': 3,
            'end_idx': -5,
            'initial_gamma': 5e-4
        }
    }
}

# Extract fitting parameters from Dataset 0 (older data)

dataset_0_temps = sorted(all_temp_corr_combined['dataset_0'].keys())
for temp in dataset_0_temps:
    print(f"\n🌡️  Dataset 0 - {temp}K:")
    temp_params = {}

    # Get stored data for this temperature from dm0
    temp_dataset = dm0.data[f'{temp}K']

    for roi_name in temp_dataset.keys():
        if roi_name != 'metadata':
            roi_data = temp_dataset[roi_name]

            # Check if fit results exist and extract parameters
            if 'fit_results' in roi_data:
                fit_result = roi_data['fit_results']

                if fit_result['success']:
                    temp_params[roi_name] = {
                        'start_idx': fit_result['start_idx'],
                        'end_idx': fit_result['end_idx'],
                        'initial_gamma': 5e-4  # Standard initial guess
                    }
                    print(f"   ✅ {roi_name}: indices [{fit_result['start_idx']}:{fit_result['end_idx']}]")
                else:
                    # Use defaults if previous fit failed
                    temp_params[roi_name] = fitting_params_combined['metadata']['default_params'].copy()
                    print(f"   ⚠️  {roi_name}: Using default indices [3:-5] (previous fit failed)")
            else:
                # Use defaults if no stored results
                temp_params[roi_name] = fitting_params_combined['metadata']['default_params'].copy()
                print(f"   ⚠️  {roi_name}: Using default indices [3:-5] (no stored results)")

    fitting_params_combined['dataset_0'][temp] = temp_params

# Extract fitting parameters from Dataset 1 (newer data)

dataset_1_temps = sorted(all_temp_corr_combined['dataset_1'].keys())
for temp in dataset_1_temps:
    print(f"\n🌡️  Dataset 1 - {temp}K:")
    temp_params = {}

    # Get stored data for this temperature from dm1
    temp_dataset = dm1.data[f'{temp}K']

    for roi_name in temp_dataset.keys():
        if roi_name != 'metadata':
            roi_data = temp_dataset[roi_name]

            # Check if fit results exist and extract parameters
            if 'fit_results' in roi_data:
                fit_result = roi_data['fit_results']

                if fit_result['success']:
                    temp_params[roi_name] = {
                        'start_idx': fit_result['start_idx'],
                        'end_idx': fit_result['end_idx'],
                        'initial_gamma': 5e-4  # Standard initial guess
                    }
                    print(f"   ✅ {roi_name}: indices [{fit_result['start_idx']}:{fit_result['end_idx']}]")
                else:
                    # Use defaults if previous fit failed
                    temp_params[roi_name] = fitting_params_combined['metadata']['default_params'].copy()
                    print(f"   ⚠️  {roi_name}: Using default indices [3:-5] (previous fit failed)")
            else:
                # Use defaults if no stored results
                temp_params[roi_name] = fitting_params_combined['metadata']['default_params'].copy()
                print(f"   ⚠️  {roi_name}: Using default indices [3:-5] (no stored results)")

    fitting_params_combined['dataset_1'][temp] = temp_params

# Summary of parameter extraction
print(f"\n✅ Fitting parameters extraction complete!")
print("=" * 60)
print(f"📊 PARAMETER SUMMARY:")
print(f"   Dataset 0: {len(fitting_params_combined['dataset_0'])} temperatures")
print(f"   Dataset 1: {len(fitting_params_combined['dataset_1'])} temperatures")

print(f"\n🎯 Data structure created: 'fitting_params_combined'")
print(f"   Access pattern: fitting_params_combined['dataset_0'][temp][roi_name]['start_idx']")
print(f"   Ready for batch fitting with optimal parameters!")

In [ ]:
# Batch fitting across both datasets with clear separation

print("Fitting for both datasets ...\n")

# Check if we have the extracted data and fitting parameters
if 'all_temp_corr_combined' not in globals():
    print("❌ No extracted correlation data found!")
    print("Please run the data extraction cell first.")
elif 'fitting_params_combined' not in globals():
    print("❌ No extracted fitting parameters found!")
    print("Please run the fitting parameters extraction cell first.")
else:
    print(f"✅ Found correlation data: {len(all_temp_corr_combined['dataset_0']) + len(all_temp_corr_combined['dataset_1'])} total temperatures")
    print(f"✅ Found fitting parameters for both datasets")


# Initialize separate storage for fitting results from both datasets
all_temp_fits_combined = {
    'dataset_0': {},  # {temp: {roi: fit_result}}
    'dataset_1': {},  # {temp: {roi: fit_result}}
    'metadata': {
        'total_fits': 0,
        'successful_fits': 0,
        'failed_fits': 0
    }
}

# Fit Dataset 0 (older data)

dataset_0_temps = sorted(all_temp_corr_combined['dataset_0'].keys())
for temp in dataset_0_temps:
    print(f"\n🌡️  Fitting Dataset 0 - {temp}K...")
    temp_fits = {}

    roi_data = all_temp_corr_combined['dataset_0'][temp]

    for roi_name in roi_data.keys():
        # Get correlation data
        F_data = roi_data[roi_name]['F'].flatten()  # Flatten to 1D if needed
        time_lags = roi_data[roi_name]['lags']

        # Get fitting parameters from extracted parameters or use defaults
        if (temp in fitting_params_combined['dataset_0'] and
            roi_name in fitting_params_combined['dataset_0'][temp]):
            params = fitting_params_combined['dataset_0'][temp][roi_name]
            start_idx = params['start_idx']
            end_idx = params['end_idx']
            initial_gamma = params['initial_gamma']
        else:
            # Fallback to defaults if parameters not found
            start_idx = 3
            end_idx = -5
            initial_gamma = 5e-4

        # Perform fitting
        try:
            fit_result = fit_F_corr(
                F_data=F_data,
                time_lags=time_lags,
                roi_name=f"D0_{roi_name}",  # Add dataset prefix for clarity
                start_idx=start_idx,
                end_idx=end_idx,
                initial_gamma=initial_gamma,
                plot=False
            )

            # Add dataset identifier to the result
            fit_result['dataset'] = 'dataset_0'
            fit_result['temperature'] = temp

            temp_fits[roi_name] = fit_result

            # Update counters
            all_temp_fits_combined['metadata']['total_fits'] += 1
            if fit_result['success']:
                all_temp_fits_combined['metadata']['successful_fits'] += 1
            else:
                all_temp_fits_combined['metadata']['failed_fits'] += 1

        except Exception as e:
            print(f"   ⚠️  {roi_name}: Error during fitting - {str(e)}")
            # Create a failed fit result for consistency
            temp_fits[roi_name] = {
                'success': False,
                'dataset': 'dataset_0',
                'temperature': temp,
                'error': str(e)
            }
            all_temp_fits_combined['metadata']['total_fits'] += 1
            all_temp_fits_combined['metadata']['failed_fits'] += 1

    all_temp_fits_combined['dataset_0'][temp] = temp_fits

# Fit Dataset 1 (newer data)

dataset_1_temps = sorted(all_temp_corr_combined['dataset_1'].keys())
for temp in dataset_1_temps:
    print(f"\n🌡️  Fitting Dataset 1 - {temp}K...")
    temp_fits = {}

    roi_data = all_temp_corr_combined['dataset_1'][temp]

    for roi_name in roi_data.keys():
        # Get correlation data
        F_data = roi_data[roi_name]['F'].flatten()  # Flatten to 1D if needed
        time_lags = roi_data[roi_name]['lags']

        # Get fitting parameters from extracted parameters or use defaults
        if (temp in fitting_params_combined['dataset_1'] and
            roi_name in fitting_params_combined['dataset_1'][temp]):
            params = fitting_params_combined['dataset_1'][temp][roi_name]
            start_idx = params['start_idx']
            end_idx = params['end_idx']
            initial_gamma = params['initial_gamma']
        else:
            # Fallback to defaults if parameters not found
            start_idx = 3
            end_idx = -5
            initial_gamma = 5e-4

        # Perform fitting
        try:
            fit_result = fit_F_corr(
                F_data=F_data,
                time_lags=time_lags,
                roi_name=f"D1_{roi_name}",  # Add dataset prefix for clarity
                start_idx=start_idx,
                end_idx=end_idx,
                initial_gamma=initial_gamma,
                plot=False
            )

            # Add dataset identifier to the result
            fit_result['dataset'] = 'dataset_1'
            fit_result['temperature'] = temp

            temp_fits[roi_name] = fit_result

            # Update counters
            all_temp_fits_combined['metadata']['total_fits'] += 1
            if fit_result['success']:
                all_temp_fits_combined['metadata']['successful_fits'] += 1
            else:
                all_temp_fits_combined['metadata']['failed_fits'] += 1

        except Exception as e:
            print(f"   ⚠️  {roi_name}: Error during fitting - {str(e)}")
            all_temp_fits_combined['metadata']['failed_fits'] += 1

    all_temp_fits_combined['dataset_1'][temp] = temp_fits

# Final summary
print(f"\n✅ CROSS-DATASET BATCH FITTING COMPLETE!")
print("=" * 60)
print(f"📊 FITTING SUMMARY:")
print(f"   Total fits attempted: {all_temp_fits_combined['metadata']['total_fits']}")
print(f"   Successful fits: {all_temp_fits_combined['metadata']['successful_fits']}")
print(f"   Failed fits: {all_temp_fits_combined['metadata']['failed_fits']}")

print(f"\n🎯 Results stored in: 'all_temp_fits_combined'")
print(f"   Dataset 0 fits: {len(all_temp_fits_combined['dataset_0'])} temperatures")
print(f"   Dataset 1 fits: {len(all_temp_fits_combined['dataset_1'])} temperatures")
print(f"   Access pattern: all_temp_fits_combined['dataset_0'][temperature][roi_name]")
print(f"   Ready for parameter comparison analysis!")

In [ ]:
# Create F(τ) comparison plots for both datasets

print("CREATING CROSS-DATASET F(τ) COMPARISON PLOTS")
print("=" * 60)

# Check if we have both correlation data and fitting results
if 'all_temp_corr_combined' not in globals() or 'all_temp_fits_combined' not in globals():
    print("❌ Missing required data!")
    print("Please run both the data extraction and batch fitting cells first.")
elif 'fitting_params_combined' not in globals():
    print("❌ Missing fitting parameters!")
    print("Please run the fitting parameters extraction cell first.")
else:
    print("✅ Found correlation data, fitting results, and fitting parameters")

# Get ROI names (assuming both datasets have the same ROIs)
if all_temp_corr_combined['dataset_0']:
    sample_temp_0 = list(all_temp_corr_combined['dataset_0'].keys())[0]
    roi_names = list(all_temp_corr_combined['dataset_0'][sample_temp_0].keys())
else:
    sample_temp_1 = list(all_temp_corr_combined['dataset_1'].keys())[0]
    roi_names = list(all_temp_corr_combined['dataset_1'][sample_temp_1].keys())

print(f"📊 Creating plots for ROIs: {roi_names}")
print(f"   Note: Faded points show excluded data, highlighted points show fit range")
print(f"   Using extracted optimal fitting parameters for each temperature/ROI")

# Get temperatures for both datasets
temps_dataset_0 = sorted(all_temp_corr_combined['dataset_0'].keys())
temps_dataset_1 = sorted(all_temp_corr_combined['dataset_1'].keys())

# Color schemes for each dataset
colors_0 = ['C0', 'C1', 'C2', 'C3', 'C4']  # Colors for dataset 0
colors_1 = ['C5', 'C6', 'C7', 'C8', 'C9', 'orange', 'purple', 'brown', 'pink']  # Colors for dataset 1

# Create four separate figures (one for each ROI)
for roi_name in roi_names:
    print(f"\n🎯 Creating plot for {roi_name}...")

    # Create figure with two subplots: left for dataset 0, right for dataset 1
    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(16, 6))

    # LEFT SUBPLOT: Dataset 0 (older data)
    ax_left.set_title(f'{roi_name} - Dataset 0 (Older Data)', fontsize=14, fontweight='bold')

    for temp_idx, temp in enumerate(temps_dataset_0):
        if temp in all_temp_corr_combined['dataset_0'] and roi_name in all_temp_corr_combined['dataset_0'][temp]:
            color = colors_0[temp_idx % len(colors_0)]

            # Get correlation data
            F_data = all_temp_corr_combined['dataset_0'][temp][roi_name]['F'].flatten()
            time_lags = all_temp_corr_combined['dataset_0'][temp][roi_name]['lags']

            # Get fitting parameters for this temperature/ROI
            if (temp in fitting_params_combined['dataset_0'] and
                roi_name in fitting_params_combined['dataset_0'][temp]):
                params = fitting_params_combined['dataset_0'][temp][roi_name]
                start_idx = params['start_idx']
                end_idx = params['end_idx'] if params['end_idx'] > 0 else len(F_data) + params['end_idx']
            else:
                start_idx = 3
                end_idx = len(F_data) - 5

            # Plot all data points (lighter/smaller for excluded points)
            ax_left.semilogx(time_lags, F_data, 'o', color=color, ms=1.5, alpha=0.3)

            # Highlight the data points used for fitting
            ax_left.semilogx(time_lags[start_idx:end_idx], F_data[start_idx:end_idx], 'o',
                           color=color, ms=2.5, alpha=0.7)

            # Plot fitted curve if available and successful
            if (temp in all_temp_fits_combined['dataset_0'] and
                roi_name in all_temp_fits_combined['dataset_0'][temp]):

                fit_result = all_temp_fits_combined['dataset_0'][temp][roi_name]
                if fit_result['success']:
                    ax_left.semilogx(fit_result['fit_times'], fit_result['fit_curve'],
                                   '-', color=color, linewidth=2.5, alpha=0.9,
                                   label=f'{temp}K fit (τ_c={fit_result["tau_char"]:.1f}s)')
                else:
                    # Show failed fit in legend
                    ax_left.plot([], [], '-', color=color, linewidth=2.5, alpha=0.5,
                               label=f'{temp}K fit (FAILED)')

    ax_left.set_xlabel('τ (s)', fontsize=12)
    ax_left.set_ylabel('F(tau)', fontsize=12)
    ax_left.set_ylim(bottom=0.97, top=1.005)
    ax_left.grid(True, alpha=0.3)
    ax_left.legend(fontsize=9, loc='best')

    # RIGHT SUBPLOT: Dataset 1 (newer data)
    ax_right.set_title(f'{roi_name} - Dataset 1 (Newer Data)', fontsize=14, fontweight='bold')

    for temp_idx, temp in enumerate(temps_dataset_1):
        if temp in all_temp_corr_combined['dataset_1'] and roi_name in all_temp_corr_combined['dataset_1'][temp]:
            color = colors_1[temp_idx % len(colors_1)]

            # Get correlation data
            F_data = all_temp_corr_combined['dataset_1'][temp][roi_name]['F'].flatten()
            time_lags = all_temp_corr_combined['dataset_1'][temp][roi_name]['lags']

            # Get fitting parameters for this temperature/ROI
            if (temp in fitting_params_combined['dataset_1'] and
                roi_name in fitting_params_combined['dataset_1'][temp]):
                params = fitting_params_combined['dataset_1'][temp][roi_name]
                start_idx = params['start_idx']
                end_idx = params['end_idx'] if params['end_idx'] > 0 else len(F_data) + params['end_idx']
            else:
                start_idx = 3
                end_idx = len(F_data) - 5

            # Plot all data points (lighter/smaller for excluded points)
            ax_right.semilogx(time_lags, F_data, 'o', color=color, ms=1.5, alpha=0.3)

            # Highlight the data points used for fitting
            ax_right.semilogx(time_lags[start_idx:end_idx], F_data[start_idx:end_idx], 'o',
                            color=color, ms=2.5, alpha=0.7)

            # Plot fitted curve if available and successful
            if (temp in all_temp_fits_combined['dataset_1'] and
                roi_name in all_temp_fits_combined['dataset_1'][temp]):

                fit_result = all_temp_fits_combined['dataset_1'][temp][roi_name]
                if fit_result['success']:
                    ax_right.semilogx(fit_result['fit_times'], fit_result['fit_curve'],
                                    '-', color=color, linewidth=2.5, alpha=0.9,
                                    label=f'{temp}K fit (τ_c={fit_result["tau_char"]:.1f}s)')
                else:
                    # Show failed fit in legend
                    ax_right.plot([], [], '-', color=color, linewidth=2.5, alpha=0.5,
                                label=f'{temp}K fit (FAILED)')

    ax_right.set_xlabel('τ (s)', fontsize=12)
    ax_right.set_ylabel('F(tau)', fontsize=12)
    ax_right.set_ylim(bottom=0.97, top=1.005)
    ax_right.grid(True, alpha=0.3)
    ax_right.legend(fontsize=9, loc='best')

    # Add overall figure title
    fig.suptitle(f'F(τ) Temperature Comparison - {roi_name}', fontsize=16, fontweight='bold')

    plt.tight_layout()
    plt.show()

print(f"\n✅ All F(τ) comparison plots created!")

In [ ]:
# Create parameter vs temperature comparison plots for both datasets

print("Creating parameter vs temperature comparison plots...")

# Extract parameters from fitting results
def extract_parameters_from_fits(fits_dict):
    """Extract tau_char and beta parameters from fit results."""
    params = {'temperatures': [], 'tau_char': [], 'beta': [], 'successful_fits': []}

    for temp in sorted(fits_dict.keys()):
        for roi_name in fits_dict[temp].keys():
            fit_result = fits_dict[temp][roi_name]
            params['temperatures'].append(temp)

            if fit_result['success']:
                params['tau_char'].append(fit_result['tau_char'])
                params['beta'].append(fit_result['beta'])
                params['successful_fits'].append(True)
            else:
                # Use NaN for failed fits (won't be plotted)
                params['tau_char'].append(np.nan)
                params['beta'].append(np.nan)
                params['successful_fits'].append(False)

    return params

# Get ROI names
if all_temp_fits_combined['dataset_0']:
    sample_temp_0 = list(all_temp_fits_combined['dataset_0'].keys())[0]
    roi_names = list(all_temp_fits_combined['dataset_0'][sample_temp_0].keys())
elif all_temp_fits_combined['dataset_1']:
    sample_temp_1 = list(all_temp_fits_combined['dataset_1'].keys())[0]
    roi_names = list(all_temp_fits_combined['dataset_1'][sample_temp_1].keys())
else:
    print("❌ No fitting results found in either dataset!")
    roi_names = []

# Extract parameters for each ROI from both datasets
roi_parameters = {}

for roi_name in roi_names:
    print(f"\nProcessing {roi_name}...")

    roi_parameters[roi_name] = {
        'dataset_0': {'temperatures': [], 'tau_char': [], 'beta': []},
        'dataset_1': {'temperatures': [], 'tau_char': [], 'beta': []}
    }

    # Extract from Dataset 0
    for temp in sorted(all_temp_fits_combined['dataset_0'].keys()):
        if roi_name in all_temp_fits_combined['dataset_0'][temp]:
            fit_result = all_temp_fits_combined['dataset_0'][temp][roi_name]
            if fit_result['success']:
                roi_parameters[roi_name]['dataset_0']['temperatures'].append(temp)
                roi_parameters[roi_name]['dataset_0']['tau_char'].append(fit_result['tau_char'])
                roi_parameters[roi_name]['dataset_0']['beta'].append(fit_result['beta'])

    # Extract from Dataset 1
    for temp in sorted(all_temp_fits_combined['dataset_1'].keys()):
        if roi_name in all_temp_fits_combined['dataset_1'][temp]:
            fit_result = all_temp_fits_combined['dataset_1'][temp][roi_name]
            if fit_result['success']:
                roi_parameters[roi_name]['dataset_1']['temperatures'].append(temp)
                roi_parameters[roi_name]['dataset_1']['tau_char'].append(fit_result['tau_char'])
                roi_parameters[roi_name]['dataset_1']['beta'].append(fit_result['beta'])

    print(f"   Dataset 0: {len(roi_parameters[roi_name]['dataset_0']['temperatures'])} successful fits")
    print(f"   Dataset 1: {len(roi_parameters[roi_name]['dataset_1']['temperatures'])} successful fits")

# Create parameter vs temperature plots
print(f"\n🎨 Creating comparison plots...")

# Colors and markers for datasets
dataset_0_color = 'C0'  # Blue
dataset_1_color = 'C1'  # Orange
dataset_0_marker = 'o'
dataset_1_marker = 's'

# Create plots for each ROI (2 parameters per ROI)
for roi_name in roi_names:
    print(f"\n📈 Creating plots for {roi_name}...")

    # Create figure with 2 subplots: tau_char and beta
    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(15, 6))

    # LEFT SUBPLOT: tau_char vs Temperature
    ax_left.set_title(f'{roi_name} - Characteristic Time τ_c', fontsize=14, fontweight='bold')

    # Plot Dataset 0
    if roi_parameters[roi_name]['dataset_0']['temperatures']:
        ax_left.plot(roi_parameters[roi_name]['dataset_0']['temperatures'],
                        roi_parameters[roi_name]['dataset_0']['tau_char'],
                        marker=dataset_0_marker, color=dataset_0_color, linewidth=2,
                        markersize=8, alpha=0.8, label='Dataset 0 (Older Data)')

    # Plot Dataset 1
    if roi_parameters[roi_name]['dataset_1']['temperatures']:
        ax_left.plot(roi_parameters[roi_name]['dataset_1']['temperatures'],
                        roi_parameters[roi_name]['dataset_1']['tau_char'],
                        marker=dataset_1_marker, color=dataset_1_color, linewidth=2,
                        markersize=8, alpha=0.8, label='Dataset 1 (Newer Data)')

    ax_left.set_xlabel('Temperature (K)', fontsize=12)
    ax_left.set_ylabel('τ_c (s)', fontsize=12)
    ax_left.grid(True, alpha=0.3)
    ax_left.legend(fontsize=11)

    # RIGHT SUBPLOT: beta vs Temperature
    ax_right.set_title(f'{roi_name} - Stretching Exponent β', fontsize=14, fontweight='bold')

    # Plot Dataset 0
    if roi_parameters[roi_name]['dataset_0']['temperatures']:
        ax_right.plot(roi_parameters[roi_name]['dataset_0']['temperatures'],
                     roi_parameters[roi_name]['dataset_0']['beta'],
                     marker=dataset_0_marker, color=dataset_0_color, linewidth=2,
                     markersize=8, alpha=0.8, label='Dataset 0 (Older Data)')

    # Plot Dataset 1
    if roi_parameters[roi_name]['dataset_1']['temperatures']:
        ax_right.plot(roi_parameters[roi_name]['dataset_1']['temperatures'],
                     roi_parameters[roi_name]['dataset_1']['beta'],
                     marker=dataset_1_marker, color=dataset_1_color, linewidth=2,
                     markersize=8, alpha=0.8, label='Dataset 1 (Newer Data)')

    ax_right.set_xlabel('Temperature (K)', fontsize=12)
    ax_right.set_ylabel('β', fontsize=12)
    ax_right.set_ylim(bottom=0.5)  # Beta should be positive
    ax_right.grid(True, alpha=0.3)
    ax_right.legend(fontsize=11)

    # Add overall figure title
    fig.suptitle(f'Parameter vs Temperature Comparison - {roi_name}', fontsize=16, fontweight='bold')

    plt.tight_layout()
    plt.show()

# Summary statistics
print(f"\n✅ PARAMETER COMPARISON ANALYSIS COMPLETE!")
print("=" * 60)
print(f"📊 SUMMARY:")

total_successful_fits_0 = sum(len(roi_parameters[roi]['dataset_0']['temperatures']) for roi in roi_names)
total_successful_fits_1 = sum(len(roi_parameters[roi]['dataset_1']['temperatures']) for roi in roi_names)

print(f"   Dataset 0 successful fits: {total_successful_fits_0}")
print(f"   Dataset 1 successful fits: {total_successful_fits_1}")
print(f"   Total successful fits: {total_successful_fits_0 + total_successful_fits_1}")

print(f"\n🎯 Generated {len(roi_names)} parameter comparison figures")
print(f"   Each figure shows τ_c (left) and β (right) vs temperature")
print(f"   Blue circles (o): Dataset 0 (older data)")
print(f"   Orange squares (s): Dataset 1 (newer data)")
print(f"   Use these plots to identify trends and compare reproducibility between datasets!")

# Notes

39.3K
* dropped first 900 frames

38.8K
* peak move

38.3K
* Intensity jumps

39K
* In-eql

38K
* Intensity jumps



## To further improve

Revive the window selection option for one-time g2&F calc

Speckle selection： Learning from Ahmad

[opencv package](https://pypi.org/project/opencv-python/)

In [ ]:
# image normalization:
normalized_array = ((img_array - img_array.min()) / (img_array.max() - img_array.min())) * 256
# Convert to integer type if needed normalized_array = normalized_array.astype(np.uint8)

In [ ]:
def speckle_finder_cv2(input_image, kernel, sd, thresh, min_size, max_size):
# Load the image
    """
    kernel should be (5,5) in this form; positive and odd
    sd is an integer
    thresh is an integer no
    """
    image = input_image #cv2.imread('scattering_image.jpg', cv2.IMREAD_GRAYSCALE)

    # Apply Gaussian Blur to reduce noise
    blurred = cv2.GaussianBlur(image, kernel, sd)
    #blurred = cv2.GaussianBlur(image, (5, 5), 0)

    # Use a binary threshold to create a binary image
    _, binary = cv2.threshold(blurred, thresh, 255, cv2.THRESH_BINARY)

    # Find contours (blobs)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Filter speckles based on size
    speckles = []
    areas = []
    for contour in contours:
        area = cv2.contourArea(contour)
        areas.append(area)
        if (area > min_size) and (area <= max_size ):  # Speckles are typically small
            speckles.append(contour)

    # Draw contours on the original image
    output = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    cv2.drawContours(output, speckles, -1, (0, 255, 0), 2)

    return output, speckles, areas

In [ ]:
def ac_calculator_speckles(speckles_list, normalized_image_in, main_df, divide_by = 1e9, deltaT = 1):
    """
    speckles_list : List of all speckles (contours); this is the output from the "identify_speckles" function
    normalized_image_in : Image normalzied from 0 to 256
    main_df: Dataframe with all data, e.g., 1did, xy coordinates, timestamps. E.g., mask_remover[0]
    divide_by: the value to divide the timestamps for binning. for 1e9 for 1s, 1e8 for 0.1s, 1e7 for 0.01s, 1e6 for 0.001s
    deltaT: each bin duration
    Outputs binned data, acfs calcualted from both MPT and Pandas Autocorrelate

    """

    #Previous version. Now we do the selection beforehand during the speckle detection phase
    #reduced_speckles = [] #select speckles that are greater than a particular area
    #for sps in speckles_list:
    #    if len(sps) > 5:
    #        reduced_speckles.append(sps)
    reduced_speckles = speckles_list
    # Since we have already chosen speckles in the previous phase; these two images are same.
    # However kept both for less hassle
    output_image_all_speckles = cv2.cvtColor(normalized_image_in.copy(), cv2.COLOR_GRAY2BGR)
    cv2.drawContours(output_image_all_speckles, speckles_list, -1, (0, 0, 255), 2)

    output_image_reduced_speckles = cv2.cvtColor(normalized_image_in.copy(), cv2.COLOR_GRAY2BGR)
    cv2.drawContours(output_image_reduced_speckles, reduced_speckles, -1, (0, 0, 255), 2)

    plt.figure(figsize = (9,6))
    #plt.subplot(121)
    #plt.imshow(output_image_all_speckles) # plt.imshow(output_image)
    #plt.title('All Speckles')
    #plt.subplot(122)
    plt.imshow(output_image_reduced_speckles)
    plt.title('All Speckles')
    plt.tight_layout()

    # now find all the pixels that are bound by the contours.
    output_image_reduced_speckles2 = output_image_reduced_speckles.copy()
    # copied the image file where the reduced contours are drawn
    speckle_coordinates_list = []              #pts_list = []
    # For each list of contour points...
    for cnt in reduced_speckles:
    # Create a mask image that contains the contour filled in
        cimg = np.zeros_like( output_image_reduced_speckles2)
        cv2.drawContours(cimg, [cnt], 0, color=255, thickness=2) # can use -1 then less points would be there
        # thickness plays a significnat role. If thickness = 2 more pixels are going to be have the value of 255.
        # Access the image pixels and create a 1D numpy array then add to list
        coords = np.where(cimg == 255)
        speckle_coordinates_list.append(coords)

    # zip xy coordinate to have format like [(x1,y1), [x2,y2], .....] and so on. Also calculate the 1did so that we can compare with dataframe.
    # Essentially, these are the speckle coordinates on which we will perform further calculations
    zipped_list_all = []
    all_1did = []
    for i in range(len(speckle_coordinates_list)):
        #(tx,ty) = (red_sp[i][:,0,1], red_sp[i][:,0,0])
        (tx,ty) = (speckle_coordinates_list[i][0], speckle_coordinates_list[i][1])
        zipped_list_all.append(list(zip(tx,ty)))
        tmp_1did = ty * 512 + tx
        all_1did.append(tmp_1did)

    tdf = main_df.copy(deep = True)   #mask_remover250[0].copy(deep = True)
    # Find the timestamps, photon counts etc. from the main dataframe for each set of speckle coordinates. Each speckle will be a dataframe that has
    # all the timestamps, coordinates etc.,. All speckle dataframes are going to be stored in a dict, e.g., spdict. For only the timestamps, we can
    # look at the tsdict. From now on, the keys in the dictionaries are going to be important because they identify the speckles and the data.
    speckles_dict = {}
    timestamps_dict = {}
    for index, ids in enumerate(all_1did):
        #print(index, ids)
        tempdf = tdf[tdf['1did'].isin(ids)].dropna()
        speckles_dict[index] = tempdf
        tempts = tempdf.explode('ts')['ts'].to_numpy().astype('int64')
        timestamps_dict[index] = tempts

    # Now, iterate through each df in the dictionary, that is iterate through each speckle, and bin the timestamps.
    binned_data_edges_dict = {}
    for key, value in timestamps_dict.items():
         #print(f"Key: {key}, Value: {value}")
         tmp_bin, tmp_edge = tpx3_2.PhotonTimeStampBinned(np.sort(value*200), divide_by)
         #binned_data_dict[key] = tmp_bin
         #bin_edges_dict[key] = tmp_edge
         temp_bin_df = pd.DataFrame({'bins': pd.Series(tmp_bin[1:-1]), 'edges': pd.Series(tmp_edge[1:-1]) }) # discard the fist and last bins
         binned_data_edges_dict[key] = temp_bin_df
    # plot the binned data, i.e., time series
    plt.figure()
    for key,val in binned_data_edges_dict.items():
        plt.plot(np.cumsum(np.diff(val['edges']))/divide_by, val['bins'].dropna()   )
    plt.show()

    # calculate acf with multipletau for each binned data, that is for each speckle
    mpt_ac_dict = {}
    for key,value in binned_data_edges_dict.items():
        tmp_ac = multipletau.autocorrelate( value['bins'].dropna().astype('float64'),
                               deltat = deltaT, normalize=True, compress = 'average')
        mpt_ac_dict[key] = tmp_ac
    # Few curves from mpt
    plt.figure()
    for key,val in mpt_ac_dict.items():
        plt.semilogx(val[:,0], val[:,1], 'o-')
    plt.title('Plots from MPT')
    plt.show()

    # Calcualte acfs with pandas autocorrelation_plot
    pandas_res_dict = {}

    for key,value in binned_data_edges_dict.items():
        tempax = autocorrelation_plot(value['bins'].dropna())
        plt.xscale('log')
        plt.title('Plot from Pandas')
        #tempax = autocorrelation_plot(value, label = "{}".format(key))
        #pandas_lags_dict[key] = tempax.lines[-1].get_xdata()
        #pandas_acs_dict[key] = tempax.lines[-1].get_ydata()
        temp_pandas_df = pd.DataFrame({'lag': tempax.lines[-1].get_xdata(), 'ac': tempax.lines[-1].get_ydata()    })
        pandas_res_dict[key] = temp_pandas_df

    #plt.figure()
    #plt.semilogx(pandas_res_dict[0]['lag'], pandas_res_dict[0]['ac'])
    #plt.semilogx(pandas_res_dict[1]['lag'], pandas_res_dict[1]['ac'])
    #[plt.axhline(y = i, linestyle = '--', color = 'gray') for i in [0.091, -0.091]]
    #[plt.axhline(y = i, linestyle = '-', color = 'gray') for i in [0.068, -0.068]]
    #plt.show()

    return binned_data_edges_dict, pandas_res_dict, mpt_ac_dict, speckles_dict, reduced_speckles

In [ ]:
# (Probably) Useful part from above

speckle_coordinates_list = []              #pts_list = []
# For each list of contour points...
for cnt in reduced_speckles:
# Create a mask image that contains the contour filled in
    cimg = np.zeros_like( output_image_reduced_speckles2)
    cv2.drawContours(cimg, [cnt], 0, color=255, thickness=2) # can use -1 then less points would be there
    # thickness plays a significnat role. If thickness = 2 more pixels are going to be have the value of 255.
    # Access the image pixels and create a 1D numpy array then add to list
    coords = np.where(cimg == 255)
    speckle_coordinates_list.append(coords)

In [ ]:
def above_line_or_not(xin, yin, xmin, xmax, hline):
    """
    Determines if the acf curve is above a confidence interval line through t-testing
    xin: lag data, yin: acf
    xmin, xmax: range of the acf curve to consider for testing
    hline: 95% or 99% CI. If there are 857 bins (as in 1s binning) the value for 95% cI
    is 1.96/np.sqrt(N = 857) = 0.068. Calculate it first and then input here. Default is 0.068
    """
    x = xin[(xin >= 0) & (xin < xmax)]
    y = yin[(xin >= 0) & (xin < xmax)]
    n = len(y)
    alpha = 0.05
    horizontal_line = hline #0.068
    y_mean = np.mean(y)
    y_std = np.std(y, ddof=1)
    t_crit = t.ppf(1 - alpha / 2, n - 1)  # Critical t-value for the confidence interval
    y_ci = t_crit * (y_std / np.sqrt(n))  # Margin of error
    # Check if the lower bound of the confidence interval is above the horizontal line
    ci_lower = y_mean - y_ci
    above_line = ci_lower > horizontal_line
    if above_line:
        return 1, y_mean, y_ci
    else:
        return 0, y_mean, y_ci

# Archived

In [ ]:
# FFT Analysis of g2 - Quality Control

print("FFT ANALYSIS OF g2")
print("=" * 60)

# Subtract the mean from g2 to remove the DC component
print(f"(g2 mean removed)\n")

# Perform FFT on the centered data
g2_fft = np.fft.fft(g2_centered)
frame_time = (exposure_time + readout_time) / 1000.0  # Convert to seconds
frequencies = np.fft.fftfreq(len(g2), d=frame_time)

# Magnitude of the Fourier Transform (consider only positive frequencies)
g2_fft_magnitude = np.abs(g2_fft)

# Take only positive frequencies
half_length = len(g2_fft_magnitude) // 2
frequencies_positive = frequencies[:half_length]
g2_fft_magnitude_positive = g2_fft_magnitude[:half_length]

# Find peaks in the Fourier Transform magnitude
threshold = 0.002        # Lower threshold to catch smaller peaks
distance = 10           # Minimum distance (number of datapoints) between peaks
prominence = 0.05         # Much lower prominence to catch visible features

peaks, _ = find_peaks(g2_fft_magnitude_positive,
                      prominence=prominence, threshold=threshold, distance=distance)

# Extract peak information
peak_frequencies = frequencies_positive[peaks]
peak_values = g2_fft_magnitude_positive[peaks]
peak_periods = 1 / peak_frequencies if len(peak_frequencies) > 0 else np.array([])

if len(peaks) > 0:
    print(f"Found {len(peaks)} significant peaks in FFT:")
    print("   Peak periods (s):", [f"{p:.2f}" for p in peak_periods])
    print("   Frequencies (Hz):", [f"{f:.4f}" for f in peak_frequencies])
else:
    print("⚠️  No peaks detected with current settings.")
    print("⚠️  Try lowering prominence or threshold if you see obvious peaks in the plot.")

# Plot FFT with identified peaks
plt.figure(figsize=(10, 6))
plt.plot(frequencies_positive, g2_fft_magnitude_positive, 'b-', linewidth=1, label="g2 FFT magnitude")

if len(peaks) > 0:
    plt.scatter(peak_frequencies, peak_values, color='red', s=50, label="Detected peaks", zorder=5)

    # Annotate peaks with periods
    for freq, period, value in zip(peak_frequencies, peak_periods, peak_values):
        plt.annotate(f"{period:.1f}s",
                    xy=(freq, value),
                    xytext=(freq + max(frequencies_positive)*0.02, value * 1.1),
                    rotation=45, color='red', fontsize=9)

plt.xlabel('Frequency (Hz)')
plt.ylabel('FFT Magnitude')
plt.title('$g_2$ Fourier Transform')
plt.yscale('log')
plt.ylim(1e-2, max(g2_fft_magnitude_positive) * 2)  # Focus plotting range above 10^(-2)
plt.grid(True, which='major', alpha=0.4, linewidth=0.8)    # Major grid
plt.grid(True, which='minor', alpha=0.2, linewidth=0.4)    # Minor grid
plt.minorticks_on()
plt.show()
